# 04. Lokale Suche: eine Lösung durch kleine Änderungen verbessern

In [Notebook 03](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/03_heuristiken_bauen.ipynb) hast du Lösungen **gebaut**: Schritt für Schritt, ohne etwas zurückzunehmen. Hier gehst du den anderen Weg: Du **startest mit einer vollständigen Lösung** (zufällig oder gierig gebaut) und **verbesserst sie durch viele kleine Änderungen**. Das ist die **lokale Suche**. Bergsteigen, Neustarts und Simulated Annealing in einfachster Form kennst du aus [Notebook 16](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/16_heuristiken.ipynb) von learn_bwinf1. Hier machst du daraus ein Werkzeug, das du auf **jede** Optimierungsaufgabe der 2. Runde anwenden kannst.

In diesem Notebook lernst du:

- eine **Bewertungsfunktion** und mehrere **Nachbarschaften** (verschieben, tauschen, größere Züge) zu entwerfen und ihre **Größe** abzuschätzen,
- die Änderung der Bewertung **inkrementell** auszurechnen (nur das neu rechnen, was der Zug berührt) und mit einem Zähler zu zeigen, wie viel das spart,
- **Bergsteigen** mit „erste Verbesserung“ und „beste Verbesserung“ und **lokale Optima** sichtbar zu machen,
- **Neustarts** und **Simulated Annealing** mit Starttemperatur, Abkühlfaktor und **Schrittbudget**,
- die **Parameter durch Experimente** zu wählen: Tabelle über mehrere Startwerte mit Mittel und Streuung, **gleiches Budget** für einen fairen Vergleich,
- eine gierige Lösung als **Start** zu nutzen und die **Güte** gegen Schranke und Optimum kleiner Fälle zu messen.

Dazu kommt eine neue Python-Technik: Du übergibst **Funktionen als Argumente** (gesammelt in einem Wörterbuch), sodass **eine** Suchfunktion für ganz verschiedene Aufgaben taugt. Das wendest du in den Ecken auf Stilvolle Päckchen (42. BwInf, mit MiniZinc als Beweis) und auf Kool (30. BwInf, für Schnelle) an.

Das neutrale Beispiel ist: **Eine Klasse in Arbeitsgruppen einteilen**. Wenn du eine Aufgabe nicht schaffst, kopierst du die Lösung aus dem aufklappbaren Teil und führst sie aus, damit die späteren Zellen laufen.

## 1. Die Aufgabe, die Bewertung und der Zustand

Eine Klasse mit `n` Schülerinnen und Schülern wird in `g` Arbeitsgruppen eingeteilt, jede Gruppe hat **höchstens `kmax`** Plätze. Für jedes Paar steht in einer Tabelle `w[i][j]`, wie gut die beiden zusammenarbeiten: eine Zahl von **+1 bis +5** heißt Wunsch (sie wollen zusammen sein), **−1 bis −5** heißt Konflikt, **0** heißt egal. Die **Bewertung** einer Einteilung ist die **Summe von `w[i][j]` über alle Paare, die in derselben Gruppe sitzen**. Sie soll **möglichst groß** werden.

Die Einteilung nennen wir den **Zustand** `z`. Weil Funktionen später **einen** Zustand übergeben bekommen, packen wir alles in ein Wörterbuch:
`z['w']` (die Tabelle), `z['g']`, `z['kmax']` und `z['gruppe']` (die Liste: `z['gruppe'][i]` ist die Gruppe von Schüler `i`, gezählt ab 0).
`pruefe_einteilung` ist der **Schiedsrichter** (wie in [Notebook 01](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/01_pruefen_und_schranken.ipynb)), `kopie` kopiert einen Zustand, `start_zufall` verteilt die Schüler zufällig, aber so, dass keine Gruppe zu voll wird. Die Zufallszahlen kommen wie in learn_bwinf1 aus `random.Random(startwert)`, damit alle dieselben Zahlen sehen.

In [ ]:
import random
import math
import statistics


def zufalls_wuensche(startwert, n):
    """Wunsch- und Konflikttabelle: w[i][j] von 1 bis 5 Wunsch, von -5 bis -1 Konflikt, 0 egal (symmetrisch)."""
    zufall = random.Random(startwert)
    w = []
    for i in range(n):
        w.append([0] * n)
    for i in range(n):
        for j in range(i):
            x = zufall.random()
            if x < 0.35:
                wert = zufall.randint(1, 5)             # Wunsch
            elif x < 0.7:
                wert = -zufall.randint(1, 5)            # Konflikt
            else:
                wert = 0
            w[i][j] = wert
            w[j][i] = wert
    return w


def neuer_zustand(w, g, kmax, gruppe):
    return {'w': w, 'g': g, 'kmax': kmax, 'gruppe': gruppe}


def kopie(z):
    return neuer_zustand(z['w'], z['g'], z['kmax'], z['gruppe'][:])


def start_zufall(w, g, kmax, zufall):
    gruppe = []
    platz = [kmax] * g
    for i in range(len(w)):
        b = zufall.randint(0, g - 1)
        while platz[b] == 0:
            b = zufall.randint(0, g - 1)
        gruppe.append(b)
        platz[b] -= 1
    return gruppe


def pruefe_einteilung(z):
    """Schiedsrichter: 'ok' oder die Beschreibung des ersten Verstoßes."""
    groesse = [0] * z['g']
    if len(z['gruppe']) != len(z['w']):
        return 'nicht jeder Schüler hat eine Gruppe'
    for i in range(len(z['gruppe'])):
        b = z['gruppe'][i]
        if b < 0 or b >= z['g']:
            return f'Schüler {i} hat die Gruppe {b}, die es nicht gibt'
        groesse[b] += 1
    for b in range(z['g']):
        if groesse[b] > z['kmax']:
            return f'Gruppe {b} hat {groesse[b]} Schüler, erlaubt sind {z["kmax"]}'
    return 'ok'


ZAEHLER = {'paare': 0}              # zählt, wie viele Paare (i, j) wir uns ansehen

Zuerst ein kleines Beispiel von Hand: sechs Schüler, zwei Gruppen mit höchstens 3 Plätzen. Zum Beispiel haben Schüler 0 und 1 den Wert +3 (sie wollen zusammen), Schüler 1 und 2 den Wert −3 (Konflikt).

In [ ]:
w6 = []
for i in range(6):
    w6.append([0] * 6)
for i, j, wert in [(0, 1, 3), (0, 2, -2), (1, 2, -3), (1, 3, 2), (2, 4, 1), (3, 5, -1), (4, 5, 2), (0, 4, 1)]:
    w6[i][j] = wert
    w6[j][i] = wert
z6 = neuer_zustand(w6, 2, 3, [0, 0, 1, 0, 1, 1])      # Gruppe 0: Schüler 0, 1, 3; Gruppe 1: Schüler 2, 4, 5
print(pruefe_einteilung(z6))
print(pruefe_einteilung(neuer_zustand(w6, 2, 3, [0, 0, 0, 0, 1, 1])))

ok
Gruppe 0 hat 4 Schüler, erlaubt sind 3


In [ ]:
def exakt(w, g, kmax):
    """Optimum durch Durchprobieren aller Einteilungen (ohne Gruppen-Vertauschungen). Nur für kleine n."""
    n = len(w)
    gruppe = [0] * n
    groesse = [0] * g
    bestes = []

    def suche(i, benutzt, wert):
        if i == n:
            if len(bestes) == 0 or wert > bestes[0]:
                bestes[:] = [wert]
            return
        for b in range(min(benutzt + 1, g)):         # höchstens eine neue Gruppe: Gruppen unterscheiden sich nur durch ihre Mitglieder
            if groesse[b] < kmax:
                gewinn = 0
                for j in range(i):
                    if gruppe[j] == b:
                        gewinn += w[i][j]
                gruppe[i] = b
                groesse[b] += 1
                suche(i + 1, max(benutzt, b + 1), wert + gewinn)
                groesse[b] -= 1

    suche(0, 0, 0)
    return bestes[0]

#### Frage 1.1

Bewerte **mit Bleistift** die Einteilung `[0, 0, 1, 0, 1, 1]` (Gruppe 0: Schüler 0, 1, 3; Gruppe 1: Schüler 2, 4, 5). Welche Paare zählen? Wie viele Paare muss man **insgesamt** ansehen, wenn man die Bewertung **ganz neu** berechnet, und wie viele bei `n = 30`?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Gruppe 0: Paar (0,1) = +3, Paar (0,3) = 0, Paar (1,3) = +2, also 5. Gruppe 1: Paar (2,4) = +1, Paar (2,5) = 0, Paar (4,5) = +2, also 3. **Summe: 8.**
Insgesamt gibt es n·(n−1)/2 Paare: bei 6 Schülern 15, bei 30 Schülern 435. Eine vollständige Bewertung sieht sich **alle** Paare an, auch wenn ein Zug nur zwei Schüler betrifft. Das merken wir uns für Abschnitt 2.
(Das Optimum dieses kleinen Beispiels ist 8; die Zelle oben hat `exakt` schon definiert, das Durchprobieren aller Einteilungen erklärt Abschnitt 8.)

</details>

#### Aufgabe 1.2

Schreibe `bewertung(z)`: die Summe von `w[i][j]` über alle Paare `i > j` mit **derselben Gruppe**. Der Zähler `ZAEHLER['paare']` soll jedes angesehene Paar zählen (die Zeile dafür steht schon da).

**Erwartete Ausgabe:**
```
8 15
```

In [ ]:
ZAEHLER['paare'] = 0


def bewertung(z):
    w = z['w']
    gruppe = z['gruppe']
    summe = 0
    for i in range(len(w)):
        for j in range(i):
            ZAEHLER['paare'] += 1
            # dein Code
    return summe


print(bewertung(z6), ZAEHLER['paare'])

<details>
<summary>Tipp anzeigen</summary>

Wenn `gruppe[i] == gruppe[j]`, addierst du `w[i][j]` zur Summe.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
ZAEHLER['paare'] = 0

def bewertung(z):
    """Summe der Werte w[i][j] über alle Paare in derselben Gruppe."""
    w = z['w']
    gruppe = z['gruppe']
    summe = 0
    for i in range(len(w)):
        for j in range(i):
            ZAEHLER['paare'] += 1
            if gruppe[i] == gruppe[j]:
                summe += w[i][j]
    return summe


print(bewertung(z6), ZAEHLER['paare'])
```

</details>

Für alle weiteren Abschnitte verwenden wir **eine große Klasse**: 30 Schüler, 5 Gruppen mit je höchstens 7 Plätzen. Die Tabelle erzeugt `zufalls_wuensche(3, 30)`. Zuerst eine **Schranke**: Mehr als die Summe aller Wünsche kann nie herauskommen (alle Wünsche erfüllt, kein Konflikt). Dann fünf Zufallsstarts als Vergleich:

In [ ]:
GROSS = zufalls_wuensche(3, 30)
G = 5                         # Gruppen
KMAX = 7                      # Plätze je Gruppe (35 Plätze für 30 Schüler)
schranke = 0
for i in range(30):
    for j in range(i):
        if GROSS[i][j] > 0:
            schranke += GROSS[i][j]
print('Schranke (alle Wünsche erfüllt, kein Konflikt):', schranke)

for startwert in range(5):
    z = neuer_zustand(GROSS, G, KMAX, start_zufall(GROSS, G, KMAX, random.Random(startwert)))
    print('Zufallsstart', startwert, pruefe_einteilung(z), bewertung(z))

Schranke (alle Wünsche erfüllt, kein Konflikt): 466
Zufallsstart 0 ok 29
Zufallsstart 1 ok 4
Zufallsstart 2 ok 32
Zufallsstart 3 ok -11
Zufallsstart 4 ok 8


#### Frage 1.3

1. Wie viele Einteilungen gibt es höchstens (jeder der 30 Schüler hat 5 Möglichkeiten)? Kann man sie alle durchprobieren?
2. Die Zufallsstarts haben die Bewertungen 29, 4, 32, -11, 8. Die Schranke ist 466. Kann man **466** erreichen? Was sagt die Schranke dann über die Güte eines Ergebnisses?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Höchstens 5³⁰ ≈ 9,3 · 10²⁰ (wegen der Platzgrenze etwas weniger, und weil man die Gruppen vertauschen darf, noch einmal etwa durch 120 teilbar; das ändert nichts Wesentliches). **Durchprobieren ist unmöglich.**
2. **Nein**: Sobald zwei Schüler, die jeweils mit einem Dritten zusammen sein wollen, **sich untereinander streiten**, kann man nicht beide Wünsche erfüllen. Dazu kommt die Platzgrenze 7. Die Schranke ist also **locker** (wie bei den Märkten in Notebook 03). Sie sagt nur: „Man kann nie mehr als 466 erreichen.“ Für die Güte brauchen wir bessere Maßstäbe: das **Optimum kleiner Fälle** (Abschnitt 8) und die beste Lösung, die lange Läufe finden.

</details>

## 2. Nachbarschaften: welche kleinen Änderungen sind erlaubt?

Die lokale Suche braucht eine **Nachbarschaft**: die Menge aller Lösungen, die man durch **eine kleine Änderung** erreicht. Wir beschreiben einen **Zug** als Tupel von Paaren `(Schüler, neue Gruppe)`:

- **Verschieben:** Ein Schüler wechselt in eine andere Gruppe, die noch Platz hat: `((i, b),)`.
- **Tauschen:** Zwei Schüler aus verschiedenen Gruppen tauschen ihre Plätze: `((i, gruppe[j]), (j, gruppe[i]))`. Die Gruppengrößen bleiben gleich.
- **Dreierring:** Drei Schüler aus drei verschiedenen Gruppen wechseln reihum: `((i, gruppe[j]), (j, gruppe[k]), (k, gruppe[i]))`. Das ist ein **größerer Zug**.

Mit einem Tupel von Paaren passt **jeder** Zug in dieselbe Funktion `anwenden(z, zug)`. Das Verschieben und den Ring bekommst du fertig (diese Zelle ausführen), das Tauschen schreibst du selbst.

In [ ]:
def zuege_verschieben(z):
    """Alle Züge 'ein Schüler wechselt in eine andere Gruppe mit freiem Platz'. Ein Zug ist ein Tupel von (Schüler, neue Gruppe)."""
    groesse = [0] * z['g']
    for b in z['gruppe']:
        groesse[b] += 1
    zuege = []
    for i in range(len(z['gruppe'])):
        for b in range(z['g']):
            if b != z['gruppe'][i] and groesse[b] < z['kmax']:
                zuege.append(((i, b),))
    return zuege


def zuege_ring(z):
    """Alle Dreierringe: Drei Schüler aus drei verschiedenen Gruppen wechseln reihum."""
    zuege = []
    gruppe = z['gruppe']
    n = len(gruppe)
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(i + 1, n):
                if len({gruppe[i], gruppe[j], gruppe[k]}) == 3:
                    zuege.append(((i, gruppe[j]), (j, gruppe[k]), (k, gruppe[i])))
    return zuege


def zuege_beide(z):
    return zuege_verschieben(z) + zuege_tauschen(z)


def anwenden(z, zug):
    for schueler, neue_gruppe in zug:
        z['gruppe'][schueler] = neue_gruppe

#### Aufgabe 2.1

Schreibe `zuege_tauschen(z)`: die Liste **aller** Züge, bei denen zwei Schüler `i` und `j` aus **verschiedenen** Gruppen ihre Plätze tauschen. Jedes Paar soll **einmal** vorkommen (nimm `j < i`). Ein Zug ist `((i, gruppe[j]), (j, gruppe[i]))`.
Darunter zählt der Test die Größe der drei Nachbarschaften für den Zufallsstart 1.

**Erwartete Ausgabe:**
```
verschieben 74
tauschen    357
Dreierring  4208
```

In [ ]:
def zuege_tauschen(z):
    zuege = []
    gruppe = z['gruppe']
    # dein Code
    return zuege


z = neuer_zustand(GROSS, G, KMAX, start_zufall(GROSS, G, KMAX, random.Random(1)))
print('verschieben', len(zuege_verschieben(z)))
print('tauschen   ', len(zuege_tauschen(z)))
print('Dreierring ', len(zuege_ring(z)))

<details>
<summary>Tipp anzeigen</summary>

Zwei Schleifen: `for i in range(n)` und `for j in range(i)`; nur wenn `gruppe[i] != gruppe[j]` ist, kommt der Zug in die Liste.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def zuege_tauschen(z):
    """Alle Züge 'zwei Schüler aus verschiedenen Gruppen tauschen die Plätze'."""
    zuege = []
    gruppe = z['gruppe']
    for i in range(len(gruppe)):
        for j in range(i):
            if gruppe[i] != gruppe[j]:
                zuege.append(((i, gruppe[j]), (j, gruppe[i])))
    return zuege


z = neuer_zustand(GROSS, G, KMAX, start_zufall(GROSS, G, KMAX, random.Random(1)))
print('verschieben', len(zuege_verschieben(z)))
print('tauschen   ', len(zuege_tauschen(z)))
print('Dreierring ', len(zuege_ring(z)))
```

</details>

#### Frage 2.2

Die drei Nachbarschaften haben bei 30 Schülern 74, 357 und 4208 Züge. Wie wächst jede Zahl mit `n` (in O-Notation, `g` ist klein und fest)? Was ändert sich ungefähr, wenn die Klasse doppelt so groß ist? Und warum ist eine **große** Nachbarschaft nicht automatisch besser?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

- **Verschieben:** höchstens n·(g−1) Züge, also **O(n)**. Doppelt so viele Schüler: etwa doppelt so viele Züge.
- **Tauschen:** höchstens n·(n−1)/2 Paare, also **O(n²)**: etwa **viermal** so viele Züge.
- **Dreierring:** drei Schüler aus n und noch zwei Drehrichtungen, also **O(n³)**: etwa **achtmal** so viele.

Eine größere Nachbarschaft enthält **mehr Verbesserungen** (ein Zustand, der kein Verschieben mehr verbessert, kann durch Tauschen noch besser werden), aber **jeder Schritt kostet mehr**, weil man mehr Nachbarn bewerten muss, um einen zu finden. Gut ist ein **Kompromiss**: oft eine kleine Nachbarschaft (billig) kombiniert mit einer zweiten (damit man aus einfachen Sackgassen herauskommt). Das probieren wir in Abschnitt 3.

</details>

### Die Bewertung inkrementell berechnen

Die Suche bewertet **sehr viele** Nachbarn. Jedes Mal die ganze Bewertung neu zu berechnen kostet n·(n−1)/2 Paare (bei n = 30: 435). Aber ein Zug berührt nur **wenige** Schüler: Die Paare, an denen **kein** gewechselter Schüler beteiligt ist, ändern ihren Beitrag nicht. Also berechnen wir nur die **Änderung** `d` der Bewertung (oft „Delta“), und zwar nur über die Paare `(i, j)`, bei denen `i` zu den gewechselten Schülern gehört:

`d` = Summe von `w[i][j] · (danach − jetzt)`, wobei `jetzt` und `danach` sagen (als 1 oder 0), ob `i` und `j` vor und nach dem Zug in derselben Gruppe sind. Wechselt auch `j`, dann ist **seine neue Gruppe** maßgebend: `neu.get(j, gruppe[j])` liefert die neue Gruppe, falls `j` wechselt, sonst die alte. Wechseln beide Schüler eines Paares, zählt das Paar nur **einmal** (bei dem kleineren Index).

#### Aufgabe 2.3

Schreibe `delta(z, zug)` nach der Beschreibung oben. Das Gerüst enthält die Schleifen und den Zähler schon, du schreibst nur, wie sich der Beitrag des Paares `(i, j)` ändert. Danach prüft ein Schiedsrichter `delta` an vielen Zügen gegen die ganz neu berechnete Bewertung und zählt die angesehenen Paare.

**Erwartete Ausgabe:**
```
delta stimmt bei allen geprüften Zügen: True
bewertung: 435 Paare
delta verschieben: 29 Paare
delta tauschen: 57 Paare
delta Dreierring: 84 Paare
```

In [ ]:
def delta(z, zug):
    w = z['w']
    gruppe = z['gruppe']
    neu = dict(zug)                          # Schüler -> neue Gruppe (nur für die, die wechseln)
    d = 0
    for i, neue_gruppe in zug:
        for j in range(len(w)):
            if j == i or (j in neu and j < i):       # Paare, bei denen beide wechseln, nur einmal zählen
                continue
            ZAEHLER['paare'] += 1
            # dein Code: jetzt = (gruppe[i] == gruppe[j]), danach = (neue_gruppe == neue Gruppe von j), d wächst um w[i][j] * (danach - jetzt)
    return d

def pruefe_delta(z, zug):
    """Schiedsrichter für delta: stimmt es mit der ganz neu berechneten Bewertung überein?"""
    vorher = bewertung(z)
    z2 = kopie(z)
    anwenden(z2, zug)
    return delta(z, zug) == bewertung(z2) - vorher


z = neuer_zustand(GROSS, G, KMAX, start_zufall(GROSS, G, KMAX, random.Random(1)))
alle_ok = True
for zuege in (zuege_verschieben(z), zuege_tauschen(z), zuege_ring(z)[:300]):
    for zug in zuege:
        if not pruefe_delta(z, zug):
            alle_ok = False
print('delta stimmt bei allen geprüften Zügen:', alle_ok)
ZAEHLER['paare'] = 0
bewertung(z)
print('bewertung:', ZAEHLER['paare'], 'Paare')
for name, zug in [('verschieben', zuege_verschieben(z)[0]), ('tauschen', zuege_tauschen(z)[0]), ('Dreierring', zuege_ring(z)[0])]:
    ZAEHLER['paare'] = 0
    delta(z, zug)
    print('delta', name + ':', ZAEHLER['paare'], 'Paare')

<details>
<summary>Tipp anzeigen</summary>

`jetzt = gruppe[i] == gruppe[j]` und `danach = neue_gruppe == neu.get(j, gruppe[j])`. Ein Wahrheitswert zählt in Rechnungen als 1 oder 0, `danach - jetzt` ist also −1, 0 oder 1.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def delta(z, zug):
    """Änderung der Bewertung durch den Zug, ohne die ganze Bewertung neu zu berechnen."""
    w = z['w']
    gruppe = z['gruppe']
    neu = dict(zug)                          # Schüler -> neue Gruppe
    d = 0
    for i, neue_gruppe in zug:
        for j in range(len(w)):
            if j == i or (j in neu and j < i):       # ein Paar, bei dem beide wechseln, nur einmal zählen
                continue
            ZAEHLER['paare'] += 1
            jetzt = gruppe[i] == gruppe[j]
            danach = neue_gruppe == neu.get(j, gruppe[j])
            d += w[i][j] * (danach - jetzt)
    return d

def pruefe_delta(z, zug):
    """Schiedsrichter für delta: stimmt es mit der ganz neu berechneten Bewertung überein?"""
    vorher = bewertung(z)
    z2 = kopie(z)
    anwenden(z2, zug)
    return delta(z, zug) == bewertung(z2) - vorher


z = neuer_zustand(GROSS, G, KMAX, start_zufall(GROSS, G, KMAX, random.Random(1)))
alle_ok = True
for zuege in (zuege_verschieben(z), zuege_tauschen(z), zuege_ring(z)[:300]):
    for zug in zuege:
        if not pruefe_delta(z, zug):
            alle_ok = False
print('delta stimmt bei allen geprüften Zügen:', alle_ok)
ZAEHLER['paare'] = 0
bewertung(z)
print('bewertung:', ZAEHLER['paare'], 'Paare')
for name, zug in [('verschieben', zuege_verschieben(z)[0]), ('tauschen', zuege_tauschen(z)[0]), ('Dreierring', zuege_ring(z)[0])]:
    ZAEHLER['paare'] = 0
    delta(z, zug)
    print('delta', name + ':', ZAEHLER['paare'], 'Paare')
```

</details>

#### Frage 2.4

Eine ganze Bewertung sieht 435 Paare an, `delta` bei einem Verschieben nur 29, bei einem Tausch 57 und bei einem Dreierring 84. Gib die Laufzeit von `bewertung` und von `delta` in O-Notation an. Wie viel Zeit spart die inkrementelle Rechnung, wenn eine Suche 100 000 Nachbarn bewertet und die Klasse 300 statt 30 Schüler hat?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

- `bewertung`: zwei verschachtelte Schleifen über alle Schüler, **O(n²)**.
- `delta`: für jeden gewechselten Schüler (höchstens 3) eine Schleife über alle anderen, **O(n)**. Die Zahl der gewechselten Schüler ist eine kleine Konstante.

Bei n = 30 spart man ungefähr den Faktor 15 (Verschieben) bis 5 (Ring). Bei n = 300 sind es 44 850 Paare gegen etwa 300 bis 900: ein Faktor von etwa 50 bis 150. Für 100 000 Nachbarn spart man also rund 4 Milliarden angesehene Paare. **Je größer die Aufgabe, desto mehr lohnt sich die inkrementelle Rechnung.** Man erkauft sich das mit einer **zweiten Formel**, die man richtig herleiten muss, deshalb prüft man sie mit einem Schiedsrichter gegen die ganze Bewertung, wie hier.

</details>

### Finde den Fehler: Tauschen falsch bewertet

Jemand schreibt `delta` mit einer Kleinigkeit anders: Für die neue Gruppe von `j` nimmt er immer `gruppe[j]`, ohne darauf zu achten, ob `j` **selbst wechselt**.

In [ ]:
def delta_falsch(z, zug):
    w = z['w']
    gruppe = z['gruppe']
    d = 0
    for i, neue_gruppe in zug:
        for j in range(len(w)):
            if j == i:
                continue
            jetzt = gruppe[i] == gruppe[j]
            danach = neue_gruppe == gruppe[j]            # Fehler: j kann selbst wechseln
            d += w[i][j] * (danach - jetzt)
    return d


z = neuer_zustand(GROSS, G, KMAX, start_zufall(GROSS, G, KMAX, random.Random(1)))
zug_v = zuege_verschieben(z)[0]
zug_t = zuege_tauschen(z)[0]
for name, zug in [('verschieben', zug_v), ('tauschen', zug_t)]:
    z2 = kopie(z)
    anwenden(z2, zug)
    print(name, 'richtig:', bewertung(z2) - bewertung(z), ' delta_falsch:', delta_falsch(z, zug))
print('Wert des Paares', zug_t[0][0], 'und', zug_t[1][0], ':', GROSS[zug_t[0][0]][zug_t[1][0]])

verschieben richtig: -9  delta_falsch: -9
tauschen richtig: -3  delta_falsch: 7
Wert des Paares 1 und 0 : 5


#### Frage 2.5

1. Warum stimmt `delta_falsch` beim Verschieben, aber nicht beim Tauschen?
2. Um wie viel liegt `delta_falsch` beim Tauschen daneben, und woher kommt dieser Wert?
3. Wie hätte man den Fehler **bemerkt**, wenn nur die Suche läuft und man `delta` nie gegen die ganze Bewertung geprüft hätte?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Beim Verschieben wechselt **nur** `i`, die anderen Schüler bleiben in ihrer Gruppe, `gruppe[j]` ist also die richtige „neue“ Gruppe von `j`. Beim Tauschen wechselt auch `j`: Der Tausch ändert die Gruppen von **beiden**, das **Paar `(i, j)` selbst** sitzt danach weiter in verschiedenen Gruppen. Die falsche Rechnung vergleicht aber `neue_gruppe` von `i` mit der **alten** Gruppe von `j`: Das ist dieselbe Gruppe, also zählt sie das Paar zu Unrecht als „zusammen“.
2. Richtig ist -3, `delta_falsch` liefert 7: der Unterschied ist **10**, der Wert des getauschten Paares (zweiter Schüler `j`, `w[i][j]` = 5). Dazu kommt es bei **jedem** Tausch, nur wenn `w[i][j]` gleich 0 ist, fällt es nicht auf.
3. Die Suche würde **Verbesserungen sehen, die keine sind**, und die gemeldete Bewertung (Startwert plus Summe aller `delta`) stimmte am Ende nicht mit `bewertung(z)` überein. Das kann man als **Schiedsrichter am Ende jeder Suche** prüfen: `wert == bewertung(z)`. Das tun alle Suchfunktionen in diesem Notebook beim Testen.

</details>

## 3. Bergsteigen: immer zu einem besseren Nachbarn

**Bergsteigen** (Hill Climbing, aus [Notebook 16](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/16_heuristiken.ipynb)): Starte mit einer Lösung und gehe immer zu einem **besseren Nachbarn**, bis es keinen mehr gibt. Dann steht man auf einem **lokalen Optimum**: einem Zustand, den kein Zug der Nachbarschaft mehr verbessert (er muss nicht das globale Optimum sein). Es gibt zwei Möglichkeiten, den nächsten Zug zu wählen:

- **Erste Verbesserung:** Gehe die Nachbarn der Reihe nach durch und nimm den **ersten**, der besser ist. Das ist billig pro Schritt.
- **Beste Verbesserung:** Bewerte **alle** Nachbarn und nimm den **besten**. Das ist teurer pro Schritt, dafür sind die Schritte größer.

Die **Nachbarschaft** wird dabei als **Funktion** übergeben: `zuege` ist eine der Funktionen `zuege_verschieben`, `zuege_tauschen`, `zuege_beide` (Aufruf `zuege(z)`). Funktionen darfst du in Python wie Zahlen als Argument übergeben, das kennst du schon aus Notebook 01. Außerdem bekommt jede Suche ein **Budget**: die Zahl der Bewertungen von Zügen (Aufrufe von `delta`), die sie höchstens ausführen darf. Das ist unser **Schrittbudget**, in Schritten statt Sekunden, damit alle dasselbe Ergebnis sehen.

#### Aufgabe 3.1

Schreibe `bergsteigen(z, zuege, beste, budget)`. Es arbeitet auf einer **Kopie** von `z` (steht schon da) und gibt `(Zustand, Wert, verbraucht)` zurück. In jeder Runde bewertet es die Züge von `zuege(z)` der Reihe nach mit `delta` und zählt `verbraucht`. Bei `beste == False` hört es beim **ersten** Zug mit `d > 0` auf zu suchen, bei `beste == True` nimmt es den Zug mit dem **größten** `d`. Hat kein Zug `d > 0`, ist es ein lokales Optimum: Ende. Am Ende jeder Runde wird der gewählte Zug mit `anwenden` ausgeführt und `wert` um `d` erhöht. Ist `verbraucht` gleich `budget`, hört die Suche sofort auf.

**Erwartete Ausgabe:**
```
erste Verbesserung: 4 -> 168 nach 901 Bewertungen, ok True
beste Verbesserung: 4 -> 168 nach 1715 Bewertungen, ok True
```

In [ ]:
def bergsteigen(z, zuege, beste, budget):
    z = kopie(z)
    wert = bewertung(z)
    verbraucht = 0
    while verbraucht < budget:
        bester_zug = None
        bestes_delta = 0
        for zug in zuege(z):
            d = delta(z, zug)
            verbraucht += 1
            # dein Code: bei d > bestes_delta merken; bei 'erste Verbesserung' (beste == False) die Schleife sofort verlassen
            if verbraucht >= budget:
                break
        if bester_zug is None:
            break                       # kein Zug verbessert: lokales Optimum
        # dein Code: den Zug ausführen und wert anpassen
    return z, wert, verbraucht


z = neuer_zustand(GROSS, G, KMAX, start_zufall(GROSS, G, KMAX, random.Random(1)))
for beste in (False, True):
    z2, wert, verbraucht = bergsteigen(z, zuege_verschieben, beste, 100000)
    print('beste' if beste else 'erste', 'Verbesserung:', bewertung(z), '->', wert, 'nach', verbraucht, 'Bewertungen,',
          pruefe_einteilung(z2), bewertung(z2) == wert)

<details>
<summary>Tipp anzeigen</summary>

Wenn `d > bestes_delta`: `bester_zug = zug` und `bestes_delta = d`; dann `if not beste: break`. Nach der Schleife: `anwenden(z, bester_zug)` und `wert += bestes_delta`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def bergsteigen(z, zuege, beste, budget):
    """Verbessert eine Kopie von z. beste=True: bester Zug, False: erster verbessernder Zug.
    Das Budget zählt die Bewertungen von Zügen (Aufrufe von delta). Gibt (Zustand, Wert, verbraucht) zurück."""
    z = kopie(z)
    wert = bewertung(z)
    verbraucht = 0
    while verbraucht < budget:
        bester_zug = None
        bestes_delta = 0
        for zug in zuege(z):
            d = delta(z, zug)
            verbraucht += 1
            if d > bestes_delta:
                bester_zug = zug
                bestes_delta = d
                if not beste:
                    break
            if verbraucht >= budget:
                break
        if bester_zug is None:
            break                           # kein Zug verbessert: lokales Optimum
        anwenden(z, bester_zug)
        wert += bestes_delta
    return z, wert, verbraucht


z = neuer_zustand(GROSS, G, KMAX, start_zufall(GROSS, G, KMAX, random.Random(1)))
for beste in (False, True):
    z2, wert, verbraucht = bergsteigen(z, zuege_verschieben, beste, 100000)
    print('beste' if beste else 'erste', 'Verbesserung:', bewertung(z), '->', wert, 'nach', verbraucht, 'Bewertungen,',
          pruefe_einteilung(z2), bewertung(z2) == wert)
```

</details>

Beide Verfahren kommen vom selben Zufallsstart auf denselben Wert, aber die „beste Verbesserung“ braucht fast doppelt so viele Bewertungen. Ist das immer so? Dafür brauchen wir **mehrere Startwerte** und Zahlen, die man vergleichen kann. Eine kleine Hilfsfunktion fasst eine Liste von Ergebnissen zusammen: **Mittel**, **Streuung** (Standardabweichung: wie weit die Werte typischerweise vom Mittel entfernt liegen), **beste** und **schlechteste** Zahl. `statistics` ist eine Bibliothek, die zur Standardbibliothek von Python gehört.

In [ ]:
def statistik(werte):
    """(Mittel, Streuung, beste, schlechteste) einer Liste von Zahlen."""
    return statistics.mean(werte), statistics.pstdev(werte), max(werte), min(werte)

Jetzt bergsteigen wir von **6 verschiedenen Zufallsstarts** (Startwerte 0 bis 5) mit allen Kombinationen aus Nachbarschaft (`verschieben`, `tauschen`, `beide` = beides zusammen) und Verbesserung (`erste`, `beste`). Das Budget ist so groß, dass jedes Bergsteigen sein lokales Optimum erreicht.

In [ ]:
print('Nachbarschaft  Verbesserung  Mittel  Streuung  beste  schlechteste  Bewertungen je Lauf')
for name, nach in [('verschieben', zuege_verschieben), ('tauschen', zuege_tauschen), ('beide', zuege_beide)]:
    for beste in (False, True):
        werte = []
        kosten = []
        for startwert in range(6):
            z = neuer_zustand(GROSS, G, KMAX, start_zufall(GROSS, G, KMAX, random.Random(startwert)))
            z2, wert, verbraucht = bergsteigen(z, nach, beste, 100000)
            werte.append(wert)
            kosten.append(verbraucht)
        mittel, streuung, bester, schlechtester = statistik(werte)
        print(f'{name:13}  {"beste" if beste else "erste":11} {mittel:7.1f} {streuung:8.1f} {bester:6d} {schlechtester:12d} {sum(kosten) / 6:12.0f}')

Nachbarschaft  Verbesserung  Mittel  Streuung  beste  schlechteste  Bewertungen je Lauf
verschieben    erste         158.8     11.6    176          144          920
verschieben    beste         160.7      9.3    168          141         1232
tauschen       erste         156.5      9.4    169          143         2921
tauschen       beste         154.3      9.0    166          142         4101
beide          erste         164.3      7.2    176          152         1488
beide          beste         167.8      8.2    177          152         5368


#### Frage 3.2

Lies die Tabelle (Werte ohne Streuung nicht interpretieren!).
1. Welche Kombination hat den besten Mittelwert, und wie viele Bewertungen kostet sie je Lauf im Vergleich zu `verschieben` mit „erste“?
2. Ist „beste Verbesserung“ besser als „erste Verbesserung“? Beachte die Streuung.
3. Was lernst du aus der Spalte „schlechteste“? Warum reicht **ein** Lauf nicht?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. `beide` mit „beste“: Mittel 167,8, aber 5368 Bewertungen je Lauf, gegen 920 bei `verschieben` mit „erste“ (Mittel 158,8). Die größere Nachbarschaft findet bessere Zustände, kostet aber mehr.
2. Nicht eindeutig: Bei allen drei Nachbarschaften liegen „erste“ und „beste“ **innerhalb der Streuung** beieinander. Die Streuung (7,2 bis 11,6) ist größer als die Unterschiede zwischen den Mittelwerten, **sechs Läufe** reichen für eine sichere Aussage nicht. „Beste“ ist pro Lauf **teurer**, ein Vorteil bei der Qualität zeigt sich hier nicht deutlich.
3. Selbst das beste Verfahren hat einen schlechtesten Lauf von 152, während der beste Lauf aller Verfahren 177 erreicht. Das Ergebnis hängt **stark vom Startpunkt** ab: Bergsteigen bleibt im ersten lokalen Optimum stecken. Deshalb: mehrere Starts, und Mittel **und** Streuung berichten.

</details>

### Lokale Optima sichtbar machen

Das Bergsteigen mit `zuege_verschieben` hört auf, wenn **kein Verschieben** mehr hilft. Aber vielleicht hilft ein **Tauschen**! Ein lokales Optimum gilt immer **für eine bestimmte Nachbarschaft**. Zähle, wie viele Züge einer Nachbarschaft einen Zustand noch verbessern.

#### Aufgabe 3.3

Schreibe `verbessernde_zuege(z, zuege)`: Sie zählt, wie viele Züge aus `zuege(z)` die Bewertung **erhöhen** (`delta > 0`). Der Test bergsteigt mit `zuege_verschieben` bis zum lokalen Optimum und zählt dann in drei Nachbarschaften.

**Erwartete Ausgabe:**
```
Startwert 0: Wert 176, verbessernde Verschiebungen 0, verbessernde Tauschzüge 0, verbessernde Dreierringe 0
Startwert 1: Wert 168, verbessernde Verschiebungen 0, verbessernde Tauschzüge 0, verbessernde Dreierringe 0
Startwert 2: Wert 144, verbessernde Verschiebungen 0, verbessernde Tauschzüge 2, verbessernde Dreierringe 4
Startwert 3: Wert 145, verbessernde Verschiebungen 0, verbessernde Tauschzüge 1, verbessernde Dreierringe 0
Startwert 4: Wert 158, verbessernde Verschiebungen 0, verbessernde Tauschzüge 1, verbessernde Dreierringe 0
Startwert 5: Wert 162, verbessernde Verschiebungen 0, verbessernde Tauschzüge 0, verbessernde Dreierringe 0
```

In [ ]:
def verbessernde_zuege(z, zuege):
    anzahl = 0
    # dein Code
    return anzahl


for startwert in range(6):
    z = neuer_zustand(GROSS, G, KMAX, start_zufall(GROSS, G, KMAX, random.Random(startwert)))
    z1, wert, verbraucht = bergsteigen(z, zuege_verschieben, False, 100000)
    print(f'Startwert {startwert}: Wert {wert}, verbessernde Verschiebungen {verbessernde_zuege(z1, zuege_verschieben)}, '
          f'verbessernde Tauschzüge {verbessernde_zuege(z1, zuege_tauschen)}, verbessernde Dreierringe {verbessernde_zuege(z1, zuege_ring)}')

<details>
<summary>Tipp anzeigen</summary>

`for zug in zuege(z)` und `if delta(z, zug) > 0:`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def verbessernde_zuege(z, zuege):
    anzahl = 0
    for zug in zuege(z):
        if delta(z, zug) > 0:
            anzahl += 1
    return anzahl


for startwert in range(6):
    z = neuer_zustand(GROSS, G, KMAX, start_zufall(GROSS, G, KMAX, random.Random(startwert)))
    z1, wert, verbraucht = bergsteigen(z, zuege_verschieben, False, 100000)
    print(f'Startwert {startwert}: Wert {wert}, verbessernde Verschiebungen {verbessernde_zuege(z1, zuege_verschieben)}, '
          f'verbessernde Tauschzüge {verbessernde_zuege(z1, zuege_tauschen)}, verbessernde Dreierringe {verbessernde_zuege(z1, zuege_ring)}')
```

</details>

#### Frage 3.4

1. Was zeigt die Ausgabe über die Bedeutung von „lokales Optimum“?
2. Du kombinierst `zuege_verschieben` und `zuege_tauschen` zu einer Nachbarschaft `zuege_beide`. Wann ist ein Zustand ein lokales Optimum für `zuege_beide`? Ist er dann auch eines für `zuege_ring`?
3. Wenn ein Zustand für **alle** drei Nachbarschaften ein lokales Optimum ist: Ist er dann das **globale** Optimum?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Ein lokales Optimum hängt von der **Nachbarschaft** ab. Alle sechs Zustände sind Optima für das Verschieben (0 verbessernde Verschiebungen), aber bei 3 von 6 gibt es noch verbessernde Tauschzüge (zusammen 4), und bei 1 von 6 verbessernde Dreierringe (zusammen 4). Eine **größere** Nachbarschaft hat **weniger** lokale Optima.
2. Wenn **weder** ein Verschieben **noch** ein Tauschen den Zustand verbessert. Für `zuege_ring` nicht automatisch: Ein Ring ist **eine andere Art von Zug**, und dort kann es noch Verbesserungen geben.
3. **Nein.** Lokal heißt nur: Kein einzelner Zug der Nachbarschaft hilft. Das globale Optimum kann weit entfernt liegen, und man müsste erst mehrere Verschlechterungen hinnehmen, um dort anzukommen. Genau dagegen helfen **Neustarts** (Abschnitt 4) und **Simulated Annealing** (Abschnitt 5).

</details>

## 4. Neustarts: mehrere Versuche mit festem Budget

Ein einzelnes Bergsteigen endet im ersten lokalen Optimum. Die einfachste Abhilfe sind **Neustarts**: Starte von einem neuen zufälligen Zustand, steige wieder bis zu einem lokalen Optimum und behalte das **beste** Ergebnis. Für einen **fairen Vergleich** bekommen alle Verfahren **dasselbe Gesamtbudget** an Bewertungen. Wir nehmen ab jetzt **3 000 Bewertungen** (`BUDGET = 3000`): Nach einem Lauf zieht man dessen Verbrauch vom Budget ab und startet neu, solange noch etwas übrig ist. Der letzte Lauf wird mit dem Rest des Budgets abgebrochen, auch wenn er noch kein lokales Optimum erreicht hat (das Ergebnis ist trotzdem ein gültiger Zustand).

#### Aufgabe 4.1

Schreibe `mit_neustarts(w, g, kmax, zuege, beste, budget, zufall)`. Die Funktion ruft `bergsteigen` immer wieder von neuen Zufallsstarts (`start_zufall(w, g, kmax, zufall)`, verpackt mit `neuer_zustand`) auf, mit **dem Rest des Budgets** als Budget. `rest` wird um `verbraucht` kleiner (mindestens um 1, damit die Schleife endet). Sie gibt `(bester Zustand, bester Wert, Zahl der Starts)` zurück.

**Erwartete Ausgabe:**
```
beide/erste, Startwert 0: 176 nach 2 Starts ok True
beide/beste, Startwert 0: 135 nach 1 Starts ok True
```

In [ ]:
def mit_neustarts(w, g, kmax, zuege, beste, budget, zufall):
    bester_z = neuer_zustand(w, g, kmax, start_zufall(w, g, kmax, zufall))     # vorläufig: ein Zufallsstart ohne Bergsteigen
    bester_wert = bewertung(bester_z)
    rest = budget
    starts = 0
    while rest > 0:
        starts += 1
        # dein Code: neuer Zufallsstart, Bergsteigen mit dem Rest als Budget, rest verkleinern, Ergebnis vergleichen
        break                       # diese Zeile löschst du, sobald dein Code `rest` verkleinert
    return bester_z, bester_wert, starts


BUDGET = 3000
z, wert, starts = mit_neustarts(GROSS, G, KMAX, zuege_beide, False, BUDGET, random.Random(0))
print('beide/erste, Startwert 0:', wert, 'nach', starts, 'Starts', pruefe_einteilung(z), bewertung(z) == wert)
z, wert, starts = mit_neustarts(GROSS, G, KMAX, zuege_beide, True, BUDGET, random.Random(0))
print('beide/beste, Startwert 0:', wert, 'nach', starts, 'Starts', pruefe_einteilung(z), bewertung(z) == wert)

<details>
<summary>Tipp anzeigen</summary>

`z = neuer_zustand(w, g, kmax, start_zufall(w, g, kmax, zufall))`, dann `z, wert, verbraucht = bergsteigen(z, zuege, beste, rest)`, `rest -= max(verbraucht, 1)` und wenn `bester_wert is None or wert > bester_wert`, merkst du dir `z` und `wert`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def mit_neustarts(w, g, kmax, zuege, beste, budget, zufall):
    """Bergsteigen von immer neuen Zufallsstarts, bis das Budget aufgebraucht ist. Gibt (bester Zustand, bester Wert, Zahl der Starts) zurück."""
    bester_z = None
    bester_wert = None
    rest = budget
    starts = 0
    while rest > 0:
        starts += 1
        z = neuer_zustand(w, g, kmax, start_zufall(w, g, kmax, zufall))
        z, wert, verbraucht = bergsteigen(z, zuege, beste, rest)
        rest -= max(verbraucht, 1)
        if bester_wert is None or wert > bester_wert:
            bester_z, bester_wert = z, wert
    return bester_z, bester_wert, starts


BUDGET = 3000
z, wert, starts = mit_neustarts(GROSS, G, KMAX, zuege_beide, False, BUDGET, random.Random(0))
print('beide/erste, Startwert 0:', wert, 'nach', starts, 'Starts', pruefe_einteilung(z), bewertung(z) == wert)
z, wert, starts = mit_neustarts(GROSS, G, KMAX, zuege_beide, True, BUDGET, random.Random(0))
print('beide/beste, Startwert 0:', wert, 'nach', starts, 'Starts', pruefe_einteilung(z), bewertung(z) == wert)
```

</details>

Jetzt der Vergleich über **8 Startwerte** (die Startwerte der Zufallszahlen für die Neustarts), jeweils mit dem Budget 3 000:

In [ ]:
print('Nachbarschaft  Verbesserung   Mittel  Streuung  beste  schlechteste  Starts je Lauf')
for name, nach in [('verschieben', zuege_verschieben), ('beide', zuege_beide)]:
    for beste in (False, True):
        werte = []
        starts_summe = 0
        for startwert in range(8):
            z, wert, starts = mit_neustarts(GROSS, G, KMAX, nach, beste, BUDGET, random.Random(startwert))
            werte.append(wert)
            starts_summe += starts
        mittel, streuung, bester, schlechtester = statistik(werte)
        print(f'{name:13}  {"beste" if beste else "erste":11} {mittel:7.1f} {streuung:8.1f} {bester:6d} {schlechtester:12d} {starts_summe / 8:12.1f}')

Nachbarschaft  Verbesserung   Mittel  Streuung  beste  schlechteste  Starts je Lauf
verschieben    erste         169.9      6.2    178          160          4.2
verschieben    beste         162.8      3.9    168          157          3.2
beide          erste         169.4      5.6    178          162          2.1
beide          beste         142.4     10.2    165          133          1.0


#### Frage 4.2

1. Warum sind die Neustarts bei `beide` mit „beste Verbesserung“ so viel schlechter als bei „erste Verbesserung“ (Mittel 142,4 gegen 169,4)? Schau auf die Spalte „Starts je Lauf“ und auf die Bewertungen je Lauf in der Tabelle von Abschnitt 3.
2. Was bedeutet das für den Vergleich zweier Verfahren, wenn man **nicht** dasselbe Budget gibt?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Ein Bergsteigen mit „beste“ und `beide` braucht im Mittel etwa 5368 Bewertungen, das **Budget von 3 000 reicht nicht einmal für einen** Lauf bis zum lokalen Optimum (im Mittel nur 1,0 Starts). Der Lauf wird **mittendrin** abgebrochen, und wir behalten einen halbfertigen Zustand. „Erste Verbesserung“ kommt mit 1488 Bewertungen aus und schafft 2,1 Starts.
2. Ein Verfahren, das **mehr Rechenzeit** bekommt, sieht oft besser aus. Ein Vergleich ist nur **fair**, wenn alle **dasselbe Budget** haben, hier gemessen in Bewertungen. (Zeit in Sekunden wäre auf jedem Rechner anders.) Dasselbe gilt für Notebook 03: „Beste von k Regeln“ kostet k-mal so viel.

</details>

## 5. Simulated Annealing: auch Verschlechterungen erlauben

Beim Bergsteigen sitzt man im lokalen Optimum fest. **Simulated Annealing** (simuliertes Abkühlen, aus [Notebook 16](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/16_heuristiken.ipynb)) probiert **zufällige Nachbarn** und nimmt

- jede Verbesserung (und jede gleich gute Änderung) **immer** an,
- eine **Verschlechterung** um `d < 0` nur mit der Wahrscheinlichkeit `e^(d/T)` (bei uns wird maximiert, also ist `d` negativ).

`T` ist die **Temperatur**: Bei großem `T` ist `e^(d/T)` nahe 1, fast alles wird angenommen (die Suche irrt umher), bei kleinem `T` fast nur noch Verbesserungen (das ist Bergsteigen). Man startet bei einer **Starttemperatur `t0`** und senkt `T` in jedem Schritt um den **Abkühlfaktor**: `T = T · faktor` mit `faktor` knapp unter 1. Der Lauf endet nach `budget` Schritten. Am Ende ist die Temperatur `t0 · faktor^budget`. Meist legt man die **Endtemperatur** fest und rechnet den Faktor aus: `faktor_fuer(t0, t_ende, budget)`.

Hier lernst du die Technik, **Funktionen als Argumente** zu übergeben, richtig kennen: Damit **eine** Funktion `abkuehlen` nicht nur Arbeitsgruppen, sondern **jede** Aufgabe bearbeiten kann, bekommt sie die **fünf Aufgabenfunktionen** in einem Wörterbuch `problem`: den Wert eines Zustands, einen zufälligen Zug, die Änderung des Wertes durch einen Zug (`delta`), das Ausführen eines Zuges und das Kopieren eines Zustands. Aufgerufen wird `problem['delta'](z, zug)`. Die Zelle liefert den zufälligen Zug (die Hälfte der Züge verschiebt, die andere Hälfte tauscht), das Wörterbuch für unsere Gruppen und `faktor_fuer`:

In [ ]:
def zufallszug(z, zufall):
    """Ein zufälliger Zug: je zur Hälfte verschieben oder tauschen."""
    n = len(z['gruppe'])
    groesse = [0] * z['g']
    for b in z['gruppe']:
        groesse[b] += 1
    while True:
        if zufall.random() < 0.5:
            i = zufall.randint(0, n - 1)
            b = zufall.randint(0, z['g'] - 1)
            if b != z['gruppe'][i] and groesse[b] < z['kmax']:
                return ((i, b),)
        else:
            i = zufall.randint(0, n - 1)
            j = zufall.randint(0, n - 1)
            if z['gruppe'][i] != z['gruppe'][j]:
                return ((i, z['gruppe'][j]), (j, z['gruppe'][i]))


PROBLEM_GRUPPEN = {'wert': bewertung, 'zufallszug': zufallszug, 'delta': delta, 'anwenden': anwenden, 'kopie': kopie}


def faktor_fuer(t0, t_ende, budget):
    """Abkühlfaktor, bei dem die Temperatur nach budget Schritten von t0 auf t_ende gesunken ist."""
    return (t_ende / t0) ** (1 / budget)

#### Aufgabe 5.1

Schreibe `annahme(d, t)`: die Wahrscheinlichkeit, mit der ein Zug mit der Änderung `d` bei der Temperatur `t` angenommen wird: **1**, wenn `d >= 0`, sonst `e^(d/t)` (`math.exp`).

**Erwartete Ausgabe:**
```
Wahrscheinlichkeit, eine Verschlechterung um d anzunehmen
T =  0.5: d = -1: 0.1353   d = -3: 0.0025   d = -10: 0.0000
T =    1: d = -1: 0.3679   d = -3: 0.0498   d = -10: 0.0000
T =    3: d = -1: 0.7165   d = -3: 0.3679   d = -10: 0.0357
T =   10: d = -1: 0.9048   d = -3: 0.7408   d = -10: 0.3679
```

In [ ]:
def annahme(d, t):
    return 0


print('Wahrscheinlichkeit, eine Verschlechterung um d anzunehmen')
for t in (0.5, 1, 3, 10):
    print(f'T = {t:4}:', '   '.join(f'd = {d}: {annahme(d, t):.4f}' for d in (-1, -3, -10)))

<details>
<summary>Tipp anzeigen</summary>

`math.exp(d / t)` ist `e` hoch `d/t`. Bei `d < 0` ist das eine Zahl zwischen 0 und 1.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def annahme(d, t):
    if d >= 0:
        return 1
    return math.exp(d / t)


print('Wahrscheinlichkeit, eine Verschlechterung um d anzunehmen')
for t in (0.5, 1, 3, 10):
    print(f'T = {t:4}:', '   '.join(f'd = {d}: {annahme(d, t):.4f}' for d in (-1, -3, -10)))
```

</details>

#### Aufgabe 5.2

Schreibe `abkuehlen(problem, z, budget, t0, faktor, zufall)` für beliebige Aufgaben. Es arbeitet auf einer Kopie von `z`, merkt sich den **besten** Zustand, den es unterwegs sieht, und gibt `(bester Zustand, bester Wert)` zurück. In jedem der `budget` Schritte:

1. einen zufälligen Zug holen und mit `delta` bewerten,
2. ihn annehmen, wenn `d >= 0` **oder** wenn `zufall.random() < math.exp(d / t)` (die Zufallszahl nur ziehen, wenn `d < 0`, damit alle dieselben Zahlen bekommen); dann den Zug ausführen, den Wert um `d` ändern und den besten Zustand aktualisieren,
3. die Temperatur um den Faktor senken.

**Erwartete Ausgabe:**
```
Start 29 -> bester Wert 176 ok True
Abkühlfaktor 0.99923 , Endtemperatur 0.3
```

In [ ]:
def abkuehlen(problem, z, budget, t0, faktor, zufall):
    z = problem['kopie'](z)
    wert = problem['wert'](z)
    bester_wert = wert
    bester_z = problem['kopie'](z)
    t = t0
    for schritt in range(budget):
        zug = problem['zufallszug'](z, zufall)
        d = problem['delta'](z, zug)
        # dein Code: annehmen? Dann ausführen, wert ändern, besten Zustand merken
        t = t * faktor
    return bester_z, bester_wert


zufall = random.Random(0)
z = neuer_zustand(GROSS, G, KMAX, start_zufall(GROSS, G, KMAX, zufall))
faktor = faktor_fuer(3, 0.3, 3000)
z2, wert = abkuehlen(PROBLEM_GRUPPEN, z, 3000, 3, faktor, zufall)
print('Start', bewertung(z), '-> bester Wert', wert, pruefe_einteilung(z2), bewertung(z2) == wert)
print('Abkühlfaktor', round(faktor, 5), ', Endtemperatur', round(3 * faktor ** 3000, 3))

<details>
<summary>Tipp anzeigen</summary>

`if d >= 0 or zufall.random() < math.exp(d / t):` dann `problem['anwenden'](z, zug)`, `wert += d` und wenn `wert > bester_wert` den Wert und eine Kopie des Zustands merken.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def abkuehlen(problem, z, budget, t0, faktor, zufall):
    """Simulated Annealing (Maximieren). Gibt (bester Zustand, bester Wert) zurück."""
    z = problem['kopie'](z)
    wert = problem['wert'](z)
    bester_wert = wert
    bester_z = problem['kopie'](z)
    t = t0
    for schritt in range(budget):
        zug = problem['zufallszug'](z, zufall)
        d = problem['delta'](z, zug)
        if d >= 0 or zufall.random() < math.exp(d / t):
            problem['anwenden'](z, zug)
            wert += d
            if wert > bester_wert:
                bester_wert = wert
                bester_z = problem['kopie'](z)
        t = t * faktor
    return bester_z, bester_wert


zufall = random.Random(0)
z = neuer_zustand(GROSS, G, KMAX, start_zufall(GROSS, G, KMAX, zufall))
faktor = faktor_fuer(3, 0.3, 3000)
z2, wert = abkuehlen(PROBLEM_GRUPPEN, z, 3000, 3, faktor, zufall)
print('Start', bewertung(z), '-> bester Wert', wert, pruefe_einteilung(z2), bewertung(z2) == wert)
print('Abkühlfaktor', round(faktor, 5), ', Endtemperatur', round(3 * faktor ** 3000, 3))
```

</details>

#### Frage 5.3

1. Die Starttemperatur ist 3. Mit welcher Wahrscheinlichkeit wird eine Verschlechterung um 3 am **Anfang** angenommen, und mit welcher bei der **Endtemperatur** 0,3? (Tabelle in Aufgabe 5.1 oder rechnen.)
2. Was passiert bei einer **sehr großen** Temperatur, was bei einer sehr kleinen (nahe 0)?
3. Warum merkt sich `abkuehlen` den **besten** Zustand und gibt nicht einfach den letzten zurück?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Anfang: e^(−3/3) = e^(−1) ≈ **0,37**, also mehr als jede dritte Verschlechterung. Ende: e^(−3/0,3) = e^(−10) ≈ **0,00005**, also praktisch nie.
2. Sehr große Temperatur: `e^(d/T)` ist nahe 1, **jeder** Zug wird angenommen, die Suche ist ein **Irrweg** ohne Richtung. Sehr kleine Temperatur: nur Verbesserungen und gleich gute Züge, also **Bergsteigen** mit zufälligem Nachbarn.
3. Auch bei niedriger Temperatur kann ein Schritt noch eine Verschlechterung annehmen. Der **letzte** Zustand ist deshalb nicht immer der beste, den man unterwegs gesehen hat. Man merkt sich den besten Zustand (Kopie!) und hat nie etwas verloren.

</details>

### Finde den Fehler: das Vorzeichen im Exponenten

Jemand schreibt die Annahme als `math.exp(-d / t)`, weil er sich an „e hoch minus etwas“ erinnert. Wir vergleichen **dieselbe Suche** mit dem richtigen und dem falschen Vorzeichen, gleicher Start, gleiches Budget:

In [ ]:
def abkuehlen_falsch(problem, z, budget, t0, faktor, zufall):
    z = problem['kopie'](z)
    wert = problem['wert'](z)
    bester_wert = wert
    t = t0
    for schritt in range(budget):
        zug = problem['zufallszug'](z, zufall)
        d = problem['delta'](z, zug)
        if d >= 0 or zufall.random() < math.exp(-d / t):         # Vorzeichen falsch!
            problem['anwenden'](z, zug)
            wert += d
            bester_wert = max(bester_wert, wert)
        t = t * faktor
    return z, bester_wert


for name, suche in [('richtig', abkuehlen), ('falsch ', abkuehlen_falsch)]:
    zufall = random.Random(0)
    z = neuer_zustand(GROSS, G, KMAX, start_zufall(GROSS, G, KMAX, zufall))
    z2, wert = suche(PROBLEM_GRUPPEN, z, 3000, 3, faktor_fuer(3, 0.3, 3000), zufall)
    print(name, ': Start', bewertung(z), ', bester Wert', wert, ', Wert am Ende', bewertung(z2))

richtig : Start 29 , bester Wert 176 , Wert am Ende 176
falsch  : Start 29 , bester Wert 77 , Wert am Ende 30


#### Frage 5.4

1. Warum führt das falsche Vorzeichen dazu, dass die Suche praktisch nichts verbessert? Rechne `math.exp(-d / t)` für ein `d = −3` und `t = 3` aus.
2. Woran hättest du es bemerkt, ohne den richtigen Code zum Vergleichen zu haben?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Für `d = −3` und `t = 3` ist `math.exp(-d / t) = math.exp(1)` ≈ 2,7, also **größer als 1**: Die Zufallszahl liegt **immer** darunter, jede Verschlechterung wird angenommen. Verbesserungen und gleich gute Züge nimmt `d >= 0` sowieso an. **Alles** wird angenommen, die Suche läuft **blind umher**: bester Wert 77 (am Ende 30) statt 176.
2. Der **Wert am Ende** ist nicht viel besser als beim Zufallsstart, obwohl die Suche lief, und er sinkt auch bei **niedriger** Temperatur nicht zu einem festen Wert zusammen. Ein Test mit **sehr kleiner** Temperatur (`t0 = 0,05`) müsste fast so gut sein wie Bergsteigen. Hier wäre das nicht der Fall. **Das Verhalten bei den Extremen (T sehr groß, T sehr klein) ist ein guter Test für jede Annahmeregel.**

</details>

## 6. Parameter durch Experiment wählen

Für Simulated Annealing musst du **Parameter** festlegen: Starttemperatur `t0`, Endtemperatur (daraus der Abkühlfaktor) und das Budget. Es gibt keine Formel, die dir die richtigen Werte sagt. Man **misst**: Man lässt dieselbe Suche mit verschiedenen Parametern über **mehrere Startwerte** laufen, mit **demselben Budget**, und liest **Mittel und Streuung** ab. Ein Unterschied zählt nur, wenn er **deutlich größer als die Streuung** ist.

Eine Faustregel für die Temperatur: Sie sollte etwa so groß sein wie die **typische Verschlechterung `d`**, die man am Anfang noch zulassen will, und am Ende **klein im Vergleich zu den kleinsten Änderungen** (hier 1). Bei unseren Werten von −5 bis +5 liegen typische Änderungen bei einigen Punkten.

#### Aufgabe 6.1

Schreibe `experiment(w, t0, t_ende, budget, startwerte)`: Für jeden Startwert wird ein Zufallsgenerator `random.Random(startwert)` angelegt, ein Zufallsstart gebaut (`start_zufall`, verpackt mit `neuer_zustand`; die Gruppenzahl 5 und die Platzgrenze 7 stehen in `G` und `KMAX`), und dann **mit demselben Zufallsgenerator** `abkuehlen(PROBLEM_GRUPPEN, ...)` mit Faktor `faktor_fuer(t0, t_ende, budget)` aufgerufen. Die Funktion gibt `statistik(werte)` der besten Werte zurück. Darunter liest du die Tabelle: nur die **Starttemperatur** wird geändert, die Endtemperatur 0,3 bleibt.

**Erwartete Ausgabe:**
```
Starttemperatur  Endtemperatur   Mittel  Streuung  beste  schlechteste
          0.3            0.3    168.0      9.1    178          152
            1            0.3    167.8      7.2    176          153
            3            0.3    173.6      3.2    177          168
           10            0.3    170.2      3.8    176          164
           30            0.3    169.9      6.1    176          157
```

In [ ]:
def experiment(w, t0, t_ende, budget, startwerte):
    werte = []
    for startwert in startwerte:
        zufall = random.Random(startwert)
        # dein Code: Zufallsstart bauen, abkühlen, den besten Wert in werte aufnehmen
    return statistik(werte if len(werte) > 0 else [0])


print('Starttemperatur  Endtemperatur   Mittel  Streuung  beste  schlechteste')
for t0 in (0.3, 1, 3, 10, 30):
    mittel, streuung, bester, schlechtester = experiment(GROSS, t0, 0.3, BUDGET, range(8))
    print(f'{t0:13}  {0.3:13}  {mittel:7.1f} {streuung:8.1f} {bester:6d} {schlechtester:12d}')

<details>
<summary>Tipp anzeigen</summary>

`z = neuer_zustand(w, G, KMAX, start_zufall(w, G, KMAX, zufall))`, dann `z2, wert = abkuehlen(PROBLEM_GRUPPEN, z, budget, t0, faktor_fuer(t0, t_ende, budget), zufall)` und `werte.append(wert)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def experiment(w, t0, t_ende, budget, startwerte):
    werte = []
    for startwert in startwerte:
        zufall = random.Random(startwert)
        z = neuer_zustand(w, G, KMAX, start_zufall(w, G, KMAX, zufall))
        z2, wert = abkuehlen(PROBLEM_GRUPPEN, z, budget, t0, faktor_fuer(t0, t_ende, budget), zufall)
        werte.append(wert)
    return statistik(werte)


print('Starttemperatur  Endtemperatur   Mittel  Streuung  beste  schlechteste')
for t0 in (0.3, 1, 3, 10, 30):
    mittel, streuung, bester, schlechtester = experiment(GROSS, t0, 0.3, BUDGET, range(8))
    print(f'{t0:13}  {0.3:13}  {mittel:7.1f} {streuung:8.1f} {bester:6d} {schlechtester:12d}')
```

</details>

Jetzt umgekehrt: Starttemperatur 3 fest, die **Endtemperatur** verändern (bei Endtemperatur 3 ist die Temperatur während des ganzen Laufs gleich 3, es wird gar nicht abgekühlt).

In [ ]:
print('Starttemperatur  Endtemperatur   Mittel  Streuung  beste  schlechteste')
for t_ende in (0.03, 0.3, 1, 3):
    mittel, streuung, bester, schlechtester = experiment(GROSS, 3, t_ende, BUDGET, range(8))
    print(f'{3:13}  {t_ende:13}  {mittel:7.1f} {streuung:8.1f} {bester:6d} {schlechtester:12d}')

Starttemperatur  Endtemperatur   Mittel  Streuung  beste  schlechteste
            3           0.03    173.6      2.0    176          169
            3            0.3    173.6      3.2    177          168
            3              1    172.5      3.1    175          166
            3              3    168.4      5.6    174          158


#### Frage 6.2

1. Welche Starttemperatur und welche Endtemperatur sind in den Tabellen am besten? Ist der Unterschied zur zweitbesten **sicher**? Vergleiche mit der Streuung.
2. Was zeigt sich bei zu **kleiner** und bei zu **großer** Starttemperatur?
3. Wie könntest du die Aussage „Temperatur 3 ist besser als Temperatur 10“ **verlässlicher** machen?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Beste Starttemperatur ist **3** (Mittel 173,6), danach kommt 10 mit 170,2. Bei der Endtemperatur sind 0,03 und 0,3 gleich gut (Mittel 173,6 und 173,6), 1 ist fast gleich (172,5). Der Abstand von 3 zu 10 (3,4 Punkte) ist **nur etwa so groß wie die Streuung** (3,2 und 3,8). **Sicher** ist die Reihenfolge der besten beiden also nicht, sicher ist nur der **Bereich**: Starttemperaturen von etwa 3 bis 10, Endtemperaturen bis etwa 1.
2. **Zu klein** (0,3 oder 1): die Suche verhält sich wie Bergsteigen und bleibt in lokalen Optima hängen. Das Mittel ist schlechter (168,0 und 167,8 gegen 173,6 bei 3), die **Streuung** ist mit 9,1 deutlich größer, und der schlechteste Lauf ist schlecht (152 gegen 168). **Zu groß** (30) oder eine Endtemperatur, die gar nicht abkühlt (3): die Suche irrt zu lange umher (Mittel 169,9 bei Starttemperatur 30; 168,4 bei Endtemperatur 3, schlechtester Lauf 158).
3. **Mehr Startwerte**: Der Fehler des Mittelwerts ist etwa Streuung geteilt durch √(Zahl der Läufe), mit 32 statt 8 Läufen halbiert er sich. Außerdem **dasselbe Budget und dieselben Startwerte** für alle Parameter benutzen, damit nur der Parameter verglichen wird.

</details>

**Kontrolle auf anderen Klassen.** Die Parameter haben wir an **einer** Klasse (`zufalls_wuensche(3, 30)`) gewählt. Wir prüfen sie an drei **neuen** Klassen (Startwerte der Tabelle 4, 5 und 6), mit je 4 Läufen und demselben Budget. Wenn die Parameter nur zufällig zu dieser einen Klasse passen, sieht man es hier:

In [ ]:
print('Klasse  Starttemperatur 0.3   Starttemperatur 3   Starttemperatur 30   (Mittel von 4 Läufen)')
for tabelle_nr in (4, 5, 6):
    andere = zufalls_wuensche(tabelle_nr, 30)
    zeile = []
    for t0 in (0.3, 3, 30):
        mittel, streuung, bester, schlechtester = experiment(andere, t0, 0.3, BUDGET, range(4))
        zeile.append(f'{mittel:19.1f}')
    print(f'{tabelle_nr:6}  ' + '   '.join(zeile))

Klasse  Starttemperatur 0.3   Starttemperatur 3   Starttemperatur 30   (Mittel von 4 Läufen)
     4                159.2                 161.2                 162.0
     5                157.5                 160.5                 160.2
     6                161.8                 164.8                 157.5


#### Frage 6.3

Gilt die Wahl der Starttemperatur auch für die neuen Klassen? Was würde sich ändern, wenn die Wünsche und Konflikte nicht von −5 bis +5, sondern von −50 bis +50 reichten?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Mit Einschränkung ja. Die Starttemperatur 3 ist bei allen drei Klassen besser als 0,3 (161,2, 160,5, 164,8 gegen 159,2, 157,5, 161,8) und bei 2 von 3 Klassen auch besser als 30 (162,0, 160,2, 157,5). Bei so wenigen Läufen (4) sind Unterschiede von einem Punkt nichts wert, der Vorsprung gegenüber der **zu kalten** Suche ist aber bei allen drei Klassen da. Die Parameter **übertragen** sich auf Klassen, die **dieselbe Größenordnung der Bewertung** haben.
Bei Werten von −50 bis +50 wären typische Änderungen **zehnmal** so groß. Dieselbe Temperatur 3 wäre dann **zehnmal zu kalt** (die Suche wäre ein Bergsteigen), man müsste die Temperaturen mit **zehn** multiplizieren. **Die Temperatur hat dieselbe Einheit wie die Bewertung**, und sie muss zu den typischen `d` passen.

</details>

**Der faire Vergleich.** Zum Schluss alle Verfahren mit demselben Budget von 3 000 Bewertungen und denselben 8 Startwerten. (Die Zelle benutzt `lambda`, eine Kurzschreibweise für eine kleine namenlose Funktion; `lambda s: ...` ist dasselbe wie `def f(s): return ...`. Du kannst sie überspringen.)

In [ ]:
print('Verfahren (Budget 3000 Bewertungen)      Mittel  Streuung  beste  schlechteste')
zeilen = []
for name, suche in [('Neustarts verschieben/erste', lambda s: mit_neustarts(GROSS, G, KMAX, zuege_verschieben, False, BUDGET, random.Random(s))[1]),
                    ('Neustarts beide/erste', lambda s: mit_neustarts(GROSS, G, KMAX, zuege_beide, False, BUDGET, random.Random(s))[1]),
                    ('Annealing 3 -> 0.3', lambda s: experiment(GROSS, 3, 0.3, BUDGET, [s])[2])]:
    werte = [suche(s) for s in range(8)]
    mittel, streuung, bester, schlechtester = statistik(werte)
    print(f'{name:38} {mittel:7.1f} {streuung:8.1f} {bester:6d} {schlechtester:12d}')

Verfahren (Budget 3000 Bewertungen)      Mittel  Streuung  beste  schlechteste
Neustarts verschieben/erste              169.9      6.2    178          160
Neustarts beide/erste                    169.4      5.6    178          162
Annealing 3 -> 0.3                       173.6      3.2    177          168


#### Frage 6.4

Annealing hat im Mittel 173,6, die Neustarts 169,4 und 169,9. Reicht das, um zu sagen, dass Annealing **immer** besser ist als Neustarts? Was sagt die Tabelle über die **Streuung** (schlechtester Lauf)?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Nein. Im Mittel liegt Annealing vorn (Abstand etwa 4,2 Punkte), aber der Abstand ist **nur etwa so groß wie die Streuung** und gilt für **diese** Klasse, diese Parameter und dieses Budget. Die **Streuung** ist bei Annealing mit 3,2 kleiner als bei den Neustarts (5,6 und 6,2), und der schlechteste Lauf ist besser (168 gegen 162 und 160): **Annealing ist verlässlicher.** Mit einem größeren Budget würde sich der Abstand noch ändern. Allgemeine Aussagen darf man nur über **das Gemessene** treffen.

</details>

## 7. Der Start: gierig oder zufällig?

Die lokale Suche braucht einen **Startzustand**. Bisher war er zufällig. Aus [Notebook 03](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/03_heuristiken_bauen.ipynb) kennst du **gierige Konstruktionen**: Sie liefern einen schon ganz guten Start. Dort hast du mehrere gierige Regeln probiert und das **beste von k** genommen. Hier nimmst du **eine** gierige Lösung und **verbesserst** sie.

Die gierige Regel für die Gruppen: Die Schüler kommen in der Reihenfolge 0, 1, 2, … dran. Jeder geht in die Gruppe mit noch freiem Platz, in der er mit den **schon eingeteilten** Schülern die **größte Summe** hat (bei Gleichstand die kleinste Gruppennummer).

#### Aufgabe 7.1

Schreibe `start_gierig(w, g, kmax)` und gib die Liste `gruppe` zurück (`gruppe[i]` ist die Gruppe von Schüler `i`). Für Schüler `i` ist der **Gewinn** in Gruppe `b` die Summe von `w[i][j]` über alle **schon eingeteilten** Schüler `j < i` mit `gruppe[j] == b`. Nur Gruppen mit freiem Platz (`platz[b] > 0`) kommen infrage.

**Erwartete Ausgabe:**
```
gierig: ok 108
Zufallsstarts: Mittel 6.0 , beste 32
```

In [ ]:
def start_gierig(w, g, kmax):
    gruppe = []
    platz = [kmax] * g
    for i in range(len(w)):
        bestes_b = None
        bester_gewinn = None
        # dein Code: die Gruppe mit dem größten Gewinn suchen
        if bestes_b is None:
            bestes_b = 0                 # nur damit das Gerüst läuft
        gruppe.append(bestes_b)
        platz[bestes_b] -= 1
    return gruppe


gruppe = start_gierig(GROSS, G, KMAX)
z_gierig = neuer_zustand(GROSS, G, KMAX, gruppe)
print('gierig:', pruefe_einteilung(z_gierig), bewertung(z_gierig))
zufallswerte = []
for startwert in range(8):
    zufallswerte.append(bewertung(neuer_zustand(GROSS, G, KMAX, start_zufall(GROSS, G, KMAX, random.Random(startwert)))))
print('Zufallsstarts: Mittel', round(sum(zufallswerte) / 8, 1), ', beste', max(zufallswerte))

<details>
<summary>Tipp anzeigen</summary>

Für jedes `b` mit `platz[b] > 0` den Gewinn ausrechnen (Schleife über `j in range(i)`). Ist er größer als `bester_gewinn` (oder ist `bester_gewinn` noch `None`), merkst du dir `b` und den Gewinn.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def start_gierig(w, g, kmax):
    """Schüler 0, 1, 2, ... kommen nacheinander in die Gruppe mit Platz, in der sie den größten Gewinn bringen."""
    gruppe = []
    platz = [kmax] * g
    for i in range(len(w)):
        bestes_b = None
        bester_gewinn = None
        for b in range(g):
            if platz[b] > 0:
                gewinn = 0
                for j in range(i):
                    if gruppe[j] == b:
                        gewinn += w[i][j]
                if bester_gewinn is None or gewinn > bester_gewinn:
                    bestes_b = b
                    bester_gewinn = gewinn
        gruppe.append(bestes_b)
        platz[bestes_b] -= 1
    return gruppe


gruppe = start_gierig(GROSS, G, KMAX)
z_gierig = neuer_zustand(GROSS, G, KMAX, gruppe)
print('gierig:', pruefe_einteilung(z_gierig), bewertung(z_gierig))
zufallswerte = []
for startwert in range(8):
    zufallswerte.append(bewertung(neuer_zustand(GROSS, G, KMAX, start_zufall(GROSS, G, KMAX, random.Random(startwert)))))
print('Zufallsstarts: Mittel', round(sum(zufallswerte) / 8, 1), ', beste', max(zufallswerte))
```

</details>

Wir vergleichen Zufallsstart und gierigen Start mit **Bergsteigen** (`beide`, erste Verbesserung) und mit **Annealing** (Budget 3 000, heiß 3 → 0,3 und kalt 0,3 → 0,3). Der gierige Start ist **immer derselbe**, bei Bergsteigen gibt es also nur **einen** Lauf (Streuung 0); bei Annealing unterscheiden sich die Läufe durch die Zufallszahlen.

In [ ]:
print('Start      Verfahren                Mittel  Streuung  beste  schlechteste')
gierig_z = neuer_zustand(GROSS, G, KMAX, start_gierig(GROSS, G, KMAX))
zeilen = []
# 1. Bergsteigen (beide, erste Verbesserung)
werte = []
for startwert in range(8):
    z = neuer_zustand(GROSS, G, KMAX, start_zufall(GROSS, G, KMAX, random.Random(startwert)))
    werte.append(bergsteigen(z, zuege_beide, False, 100000)[1])
zeilen.append(('zufällig', 'Bergsteigen', werte))
zeilen.append(('gierig', 'Bergsteigen', [bergsteigen(gierig_z, zuege_beide, False, 100000)[1]]))
# 2. Annealing mit Budget 3000: heiß (3 -> 0.3) und kalt (0.3 -> 0.3)
for t0 in (3, 0.3):
    for start in ('zufällig', 'gierig'):
        werte = []
        for startwert in range(8):
            zufall = random.Random(startwert)
            if start == 'zufällig':
                z = neuer_zustand(GROSS, G, KMAX, start_zufall(GROSS, G, KMAX, zufall))
            else:
                z = gierig_z
            werte.append(abkuehlen(PROBLEM_GRUPPEN, z, BUDGET, t0, faktor_fuer(t0, 0.3, BUDGET), zufall)[1])
        zeilen.append((start, 'Annealing ' + str(t0) + ' -> 0.3', werte))
for start, name, werte in zeilen:
    mittel, streuung, bester, schlechtester = statistik(werte)
    print(f'{start:9}  {name:22} {mittel:7.1f} {streuung:8.1f} {bester:6d} {schlechtester:12d}')

Start      Verfahren                Mittel  Streuung  beste  schlechteste
zufällig   Bergsteigen              165.2      7.1    176          152
gierig     Bergsteigen              176.0      0.0    176          176
zufällig   Annealing 3 -> 0.3       173.6      3.2    177          168
gierig     Annealing 3 -> 0.3       173.2      5.7    178          160
zufällig   Annealing 0.3 -> 0.3     168.0      9.1    178          152
gierig     Annealing 0.3 -> 0.3     168.5      5.9    175          160


#### Frage 7.2

1. Der gierige Start hat 108 Punkte, die Zufallsstarts im Mittel 6,0. Hilft der bessere Start beim **Bergsteigen**? Wie sicher ist die Aussage?
2. Beim Annealing macht der Start fast keinen Unterschied, weder bei der heißen Suche (Mittel 173,2 gegen 173,6) noch bei der kalten (Mittel 168,5 gegen 168,0). Warum nicht bei der heißen? Und warum wohl auch nicht bei der kalten, obwohl die gierige Lösung viel besser ist als ein Zufallsstart?
3. Wann würdest du einen gierigen Start nehmen, wann einen zufälligen?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Hier scheint es so: Vom gierigen Start aus erreicht Bergsteigen 176, von den Zufallsstarts im Mittel 165,2. Aber der gierige Start liefert **nur einen** Lauf, und der **beste** der acht Zufallsläufe erreicht ebenfalls 176. Wir haben **eine** Zahl und **keine Streuung**, die Aussage ist also nicht sicher. Außerdem muss man den gierigen Start erst entwerfen, und er ist **nicht immer** besser (eine schlechte gierige Regel führt in ein schlechtes lokales Optimum).
2. Bei Temperatur 3 werden anfangs **fast alle** Züge angenommen: Die gute Startlösung wird in den ersten Schritten **zerstört**, und der Lauf vergisst, wo er angefangen hat. Bei der kalten Suche bleibt der Start zwar erhalten, aber die gierige Lösung liegt mit 108 Punkten noch **weit unter** dem Niveau der lokalen Optima (rund 170), und die 3 000 Schritte reichen, um dieses Niveau auch vom Zufallsstart aus zu erreichen. **Ein gieriger Start lohnt sich vor allem, wenn das Budget knapp ist.**
3. Zufallsstart: wenn die Suche **heiß** startet, wenn man **viele unabhängige** Läufe will oder wenn es keine gute gierige Regel gibt. Gieriger Start: wenn das Budget **knapp** ist oder die Aufgabe sehr groß (die lokale Suche bräuchte sonst viele Schritte, um erst in die Nähe einer guten Lösung zu kommen), dann mit **kalter** Suche. Hat man mehrere gierige Regeln (Notebook 03), startet man von jeder und behält das beste Ergebnis.

</details>

## 8. Wie gut ist das Ergebnis? Die Güte messen

Wie in [Notebook 01](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/01_pruefen_und_schranken.ipynb) und 03 gehört zu jeder Heuristik eine **Aussage über die Güte**. Bei den Arbeitsgruppen haben wir zwei Maßstäbe:

- **Kleine Klassen:** das **Optimum** durch Durchprobieren. `exakt` (steht in der Zelle in Abschnitt 1) probiert alle Einteilungen, aber ohne **Gruppen-Vertauschungen**: Die Gruppen unterscheiden sich nur durch ihre Mitglieder, also darf ein Schüler nur in eine **schon benutzte** Gruppe oder in die **nächste neue** gehen. Das spart den Faktor g!. Für etwa 10 Schüler geht das schnell.
- **Große Klassen:** die **Schranke** (Summe aller Wünsche) ist locker. Besser ist die **beste Lösung, die lange Läufe finden**. Das ist **kein Beweis**, aber eine ehrliche Untergrenze für das Optimum.

Wir testen auf acht kleinen Klassen mit 10 Schülern, 3 Gruppen und höchstens 4 Plätzen und einem **kleinen Budget** (150 Bewertungen), damit man Unterschiede sieht. Spalten: Optimum und die Ergebnisse von Bergsteigen (`beide`, erste Verbesserung, ein Zufallsstart), Neustarts (`beide`, erste Verbesserung) und Annealing (Starttemperatur 3, Endtemperatur 0,3).

In [ ]:
KLEINE = {}
for nummer in range(41, 49):
    w = zufalls_wuensche(nummer, 10)
    KLEINE[nummer] = (w, exakt(w, 3, 4))          # Optimum durch Durchprobieren


def vergleich(budget, tabelle):
    treffer = [0, 0, 0]
    summe_guete = [0, 0, 0]
    if tabelle:
        print('Klasse  Optimum  Bergsteigen  Neustarts  Annealing')
    for nummer in KLEINE:
        w, opt = KLEINE[nummer]
        zufall = random.Random(nummer)
        z = neuer_zustand(w, 3, 4, start_zufall(w, 3, 4, zufall))
        erg_b = bergsteigen(z, zuege_beide, False, budget)[1]
        erg_n = mit_neustarts(w, 3, 4, zuege_beide, False, budget, random.Random(nummer))[1]
        zufall = random.Random(nummer)
        z = neuer_zustand(w, 3, 4, start_zufall(w, 3, 4, zufall))
        erg_a = abkuehlen(PROBLEM_GRUPPEN, z, budget, 3, faktor_fuer(3, 0.3, budget), zufall)[1]
        for k, erg in enumerate((erg_b, erg_n, erg_a)):
            if erg == opt:
                treffer[k] += 1
            summe_guete[k] += 100 * erg / opt
        if tabelle:
            print(f'{nummer:6}  {opt:7}  {erg_b:11}  {erg_n:9}  {erg_a:9}')
    print(f'Budget {budget}: optimal erreicht (Bergsteigen, Neustarts, Annealing):', *treffer, 'von 8;',
          'Güte im Mittel in Prozent:', *(round(s / 8, 1) for s in summe_guete))


vergleich(150, True)
vergleich(1500, False)

Klasse  Optimum  Bergsteigen  Neustarts  Annealing
    41       14           12         13         13
    42       32           30         32         29
    43       29           29         29         29
    44       27           25         27         27
    45       32           32         32         31
    46       17           17         17         17
    47       24           20         20         20
    48       40           38         38         38
Budget 150: optimal erreicht (Bergsteigen, Neustarts, Annealing): 3 5 3 von 8; Güte im Mittel in Prozent: 93.8 96.4 94.8
Budget 1500: optimal erreicht (Bergsteigen, Neustarts, Annealing): 3 8 8 von 8; Güte im Mittel in Prozent: 93.8 100.0 100.0


#### Frage 8.1

1. Mit dem kleinen Budget 150 erreichen Bergsteigen, Neustarts und Annealing das Optimum in 3, 5 und 3 von 8 Klassen. Welche Aussage über die Güte der drei Verfahren ist **gesichert**, welche nicht?
2. Mit dem Budget 1 500 sind es 3, 8 und 8 von 8. Was lernst du daraus?
3. Bei der großen Klasse (30 Schüler) ist die Schranke 466, und die beste Bewertung, die wir in diesem Notebook gesehen haben, ist 178 (Neustarts, Budget 3 000). Was kannst du über die Güte der Ergebnisse sagen?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. **Gesichert** ist nur: Auf diesen 8 Klassen mit 10 Schülern erreichen die Neustarts bei Budget 150 im Mittel 96,4 % des Optimums, Annealing 94,8 % und einmaliges Bergsteigen 93,8 %. Der schlechteste Einzelfall beim Annealing ist 83,3 %. **Nicht** gesichert: dass Neustarts bei **größeren** Klassen oder anderem Budget auch besser sind als Annealing. Bei diesem winzigen Budget (150 Schritte, 10 Schüler) bleibt Annealing wenig Zeit zum Abkühlen.
2. Bei Budget 1 500 sind **Neustarts und Annealing bei allen 8 Klassen optimal**. Das Budget entscheidet über die Güte, mehr Schritte helfen. Auf kleinen Klassen reicht ein bescheidenes Budget, **die Güte bei großen Klassen muss man eigens messen**.
3. Die Schranke (466) ist **locker**: Selbst das beste Ergebnis liegt weit darunter. In vier langen Annealing-Läufen mit je 40 000 Schritten fanden wir höchstens 178. Das ist **kein Beweis** für das Optimum, aber ein guter Hinweis: Die Ergebnisse mit Budget 3 000 (Mittel bis 173,6) liegen nur wenig unter 178. Ehrlich sagt man: „178 ist die beste bekannte Bewertung, unsere Ergebnisse erreichen im Mittel etwa 97,5 % davon.“ Wer das Optimum **beweisen** will, braucht einen Solver; ein kleines Modell dieser Art zeigt der Abschnitt „Zum Schluss“ bei den Päckchen.

</details>

## Wettbewerbs-Ecke 1: Stilvolle Päckchen

*Angelehnt an Aufgabe 2 „Stilvolle Päckchen“ der 2. Runde des 42. Bundeswettbewerbs Informatik (2023/24).*

Aus einem Bestand an Kleidungsstücken (jedes hat eine **Sorte**, etwa Hose oder Jacke, und einen **Stil**, etwa sportlich oder formell) sollen **Outfit-Boxen** gepackt werden. In jeder Box ist von **jeder Sorte** mindestens ein und höchstens drei Stücke, und **alle Stile** in der Box müssen **paarweise zusammenpassen**. Möglichst **wenige Stücke** sollen übrig bleiben. Das Lösungsnotebook [Stilvolle Päckchen](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/42_StilvollePaeckchen/paeckchen_loesung.ipynb) beschreibt die Aufgabe genauer.

Wir vereinfachen so, dass die **lokale Suche** gut passt. Eine Erkenntnis aus dem Lösungsnotebook lassen wir uns schenken: Die Stile einer Box bilden eine **Clique** (alle passen paarweise zusammen), und jede Box lässt sich einer **maximalen Clique** zuordnen. Für jede Clique gilt: Hat sie `n[s]` Stücke der Sorte `s`, dann kann man `m = min(n[s])` Boxen packen und von jeder Sorte höchstens `3·m` Stücke verbrauchen. **Gepackt** werden also `Σ min(n[s], 3·m)` Stücke. Die Dateien `daten/paeckchen_cliquen3.txt` und `daten/paeckchen_cliquen5.txt` (aus den Beispielen `paeckchen3` und `paeckchen5` mit 96 und 174 Stücken) enthalten schon die **maximalen Cliquen**.

Die Aufgabe heißt jetzt: **Ordne jedes Kleidungsstück einer Clique zu (oder lass es übrig), sodass möglichst viele Stücke gepackt werden.** Der **Zustand** ist die Zuordnung, der **Wert** ist die Summe der `wert_clique`-Werte aller Cliquen, ein **Zug** verschiebt **ein** Stück `(Sorte s, Stil r)` aus seiner Clique `a` in eine Clique `b`, in der der Stil `r` vorkommt, oder nach „übrig“. Die Bewertung ändert sich nur bei den **zwei** beteiligten Cliquen: ideal für die inkrementelle Rechnung! Die Funktion `abkuehlen` aus Abschnitt 5 passt **ohne Änderung**, weil sie nur die fünf Aufgabenfunktionen aufruft.

Zuerst die Bewertung einer **Clique**: `n_je_sorte[s]` ist die Zahl der Stücke der Sorte `s` in der Clique. Ist ein Wert 0, kann man keine Box packen. Sonst ist `m` das Kleinste und es werden von jeder Sorte höchstens `3·m` Stücke gepackt.

#### Aufgabe E1.1

Schreibe `wert_clique(n_je_sorte)`: `m` ist das Kleinste der Zahlen, gepackt werden `min(n, 3 * m)` Stücke jeder Sorte, `wert_clique` ist die Summe. Beispiel: `[4, 5, 3]` hat `m = 3`, also `min(4, 9) + min(5, 9) + min(3, 9) = 12`; `[2, 7, 1]` hat `m = 1`, also `2 + 3 + 1 = 6`.

**Erwartete Ausgabe:**
```
0 12 6
```

In [ ]:
def wert_clique(n_je_sorte):
    m = min(n_je_sorte)
    summe = 0
    # dein Code
    return summe


print(wert_clique([3, 2, 0]), wert_clique([4, 5, 3]), wert_clique([2, 7, 1]))

<details>
<summary>Tipp anzeigen</summary>

`for n in n_je_sorte: summe += min(n, 3 * m)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def wert_clique(n_je_sorte):
    """Zahl der Stücke, die man mit n_je_sorte[s] Stücken je Sorte in Boxen packen kann."""
    m = min(n_je_sorte)
    summe = 0
    for n in n_je_sorte:
        summe += min(n, 3 * m)
    return summe


print(wert_clique([3, 2, 0]), wert_clique([4, 5, 3]), wert_clique([2, 7, 1]))
```

</details>

In [ ]:
def lies_cliquen(dateiname):
    """Datei: 'Sorten Stile K', dann K Zeilen mit den Stilen einer maximalen Clique, eine Leerzeile,
    dann je Sorte eine Zeile mit der Zahl der Stücke je Stil. Stile werden ab 0 gezählt."""
    with open(dateiname, encoding='utf-8') as datei:
        zeilen = datei.read().split('\n')
    sorten, stile, k = [int(x) for x in zeilen[0].split()]
    cliquen = []
    for zeile in zeilen[1:1 + k]:
        cliquen.append([int(x) - 1 for x in zeile.split()])
    anzahl = []
    for zeile in zeilen[2 + k:2 + k + sorten]:
        anzahl.append([int(x) for x in zeile.split()])
    return sorten, stile, cliquen, anzahl


def neuer_zustand_paeckchen(dateiname, zufall):
    """Zufällige Zuordnung: jedes Stück in eine zufällige Clique, zu der sein Stil gehört. Der Index K steht für 'übrig'."""
    sorten, stile, cliquen, anzahl = lies_cliquen(dateiname)
    k = len(cliquen)
    z = {'S': sorten, 'K': k, 'erlaubt': {}, 'n': [], 'zuordnung': {}, 'stuecke': [], 'gesamt': 0}
    for a in range(k):
        z['n'].append([0] * sorten)                      # z['n'][a][s]: Stücke der Sorte s in Clique a
    for s in range(sorten):
        for r in range(stile):
            if anzahl[s][r] > 0:
                z['erlaubt'][(s, r)] = [a for a in range(k) if r in cliquen[a]] + [k]
                z['zuordnung'][(s, r)] = [0] * (k + 1)    # wie viele Stücke (s, r) in welcher Clique liegen
                z['stuecke'].append((s, r))
                z['gesamt'] += anzahl[s][r]
                for _ in range(anzahl[s][r]):
                    a = zufall.choice(z['erlaubt'][(s, r)][:-1])
                    z['zuordnung'][(s, r)][a] += 1
                    z['n'][a][s] += 1
    return z

def wert_paeckchen(z):
    summe = 0
    for a in range(z['K']):
        summe += wert_clique(z['n'][a])
    return summe


def zufallszug_paeckchen(z, zufall):
    """Ein zufälliges Stück (Sorte s, Stil r) aus seiner Clique a (oder aus 'übrig') in eine andere erlaubte Clique b."""
    while True:
        s, r = zufall.choice(z['stuecke'])
        a = zufall.choice([k for k in range(z['K'] + 1) if z['zuordnung'][(s, r)][k] > 0])
        b = zufall.choice(z['erlaubt'][(s, r)])
        if a != b:
            return (s, r, a, b)


def anwenden_paeckchen(z, zug):
    s, r, a, b = zug
    z['zuordnung'][(s, r)][a] -= 1
    z['zuordnung'][(s, r)][b] += 1
    if a < z['K']:
        z['n'][a][s] -= 1
    if b < z['K']:
        z['n'][b][s] += 1


def kopie_paeckchen(z):
    neu = dict(z)
    neu['n'] = [zeile[:] for zeile in z['n']]
    neu['zuordnung'] = {schluessel: zeile[:] for schluessel, zeile in z['zuordnung'].items()}
    return neu

Die Zelle oben liest die Datei, baut einen **Zufallszustand** und definiert die Funktionen `wert_paeckchen`, `zufallszug_paeckchen`, `anwenden_paeckchen` und `kopie_paeckchen`. Ein **Zug** ist `(s, r, a, b)`: ein Stück der Sorte `s` und des Stils `r` wechselt aus der Clique `a` in die Clique `b`; der Index `K` (die Zahl der Cliquen) bedeutet „übrig“, dort zählt das Stück nichts.
Was fehlt, ist die **Änderung der Bewertung** `delta_paeckchen`, und die hängt nur von den zwei beteiligten Cliquen ab.

#### Aufgabe E1.2

Schreibe `delta_paeckchen(z, zug)` für einen Zug `(s, r, a, b)`. Ist `a < z['K']`, verliert die Clique `a` ein Stück der Sorte `s`: Rechne `wert_clique(z['n'][a])` vorher, ziehe 1 von `z['n'][a][s]` ab, rechne nachher und **stelle den Zustand wieder her** (`+= 1`). Ist `b < z['K']`, gewinnt die Clique `b` ein Stück: ebenso mit `+ 1`. `d` ist die Summe der beiden Änderungen. Der Schiedsrichter prüft 300 zufällige Züge gegen die ganz neu berechnete Bewertung (er führt die Züge aus, also ändert sich der Wert).

**Erwartete Ausgabe:**
```
delta stimmt bei 300 Zügen: True , Stücke: 96 , Wert: 37
```

In [ ]:
def delta_paeckchen(z, zug):
    s, r, a, b = zug
    d = 0
    if a < z['K']:
        # dein Code
        pass
    if b < z['K']:
        # dein Code
        pass
    return d


PROBLEM_PAECKCHEN = {'wert': wert_paeckchen, 'zufallszug': zufallszug_paeckchen, 'delta': delta_paeckchen,
                     'anwenden': anwenden_paeckchen, 'kopie': kopie_paeckchen}

z = neuer_zustand_paeckchen('daten/paeckchen_cliquen3.txt', random.Random(0))
zufall = random.Random(1)
ok = True
for _ in range(300):
    zug = zufallszug_paeckchen(z, zufall)
    d = delta_paeckchen(z, zug)
    vorher = wert_paeckchen(z)
    anwenden_paeckchen(z, zug)
    ok = ok and d == wert_paeckchen(z) - vorher
print('delta stimmt bei 300 Zügen:', ok, ', Stücke:', z['gesamt'], ', Wert:', wert_paeckchen(z))

<details>
<summary>Tipp anzeigen</summary>

Für `a`: `vorher = wert_clique(z['n'][a])`, `z['n'][a][s] -= 1`, `d += wert_clique(z['n'][a]) - vorher`, `z['n'][a][s] += 1`. Für `b` genauso mit `+= 1` und `-= 1` vertauscht.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def delta_paeckchen(z, zug):
    s, r, a, b = zug
    d = 0
    if a < z['K']:
        vorher = wert_clique(z['n'][a])
        z['n'][a][s] -= 1
        d += wert_clique(z['n'][a]) - vorher
        z['n'][a][s] += 1
    if b < z['K']:
        vorher = wert_clique(z['n'][b])
        z['n'][b][s] += 1
        d += wert_clique(z['n'][b]) - vorher
        z['n'][b][s] -= 1
    return d


PROBLEM_PAECKCHEN = {'wert': wert_paeckchen, 'zufallszug': zufallszug_paeckchen, 'delta': delta_paeckchen,
                     'anwenden': anwenden_paeckchen, 'kopie': kopie_paeckchen}

z = neuer_zustand_paeckchen('daten/paeckchen_cliquen3.txt', random.Random(0))
zufall = random.Random(1)
ok = True
for _ in range(300):
    zug = zufallszug_paeckchen(z, zufall)
    d = delta_paeckchen(z, zug)
    vorher = wert_paeckchen(z)
    anwenden_paeckchen(z, zug)
    ok = ok and d == wert_paeckchen(z) - vorher
print('delta stimmt bei 300 Zügen:', ok, ', Stücke:', z['gesamt'], ', Wert:', wert_paeckchen(z))
```

</details>

Jetzt rufen wir **dieselbe** Funktion `abkuehlen` wie bei den Arbeitsgruppen auf, mit dem Wörterbuch `PROBLEM_PAECKCHEN`. Die **übrigen** Stücke sind `gesamt − Wert`. Wir vergleichen eine **kalte** Suche (Temperatur immer 0,05, praktisch Bergsteigen) mit einer **heißen** (Start bei 1, Ende bei 0,05), je 8 Startwerte, Budget 3 000:

In [ ]:
print('Übrige Stücke je Startwert 0 bis 7 (Budget 3000):')
for datei in ('daten/paeckchen_cliquen3.txt', 'daten/paeckchen_cliquen5.txt'):
    for name, t0 in [('kalt 0.05 -> 0.05', 0.05), ('heiß 1 -> 0.05', 1)]:
        uebrig = []
        for startwert in range(8):
            zufall = random.Random(startwert)
            z = neuer_zustand_paeckchen(datei, zufall)
            faktor = faktor_fuer(t0, 0.05, 3000)
            z2, wert = abkuehlen(PROBLEM_PAECKCHEN, z, 3000, t0, faktor, zufall)
            uebrig.append(z['gesamt'] - wert)
        print(f'{datei[-12:-4]}  {name:18}', uebrig, ' kleinster Wert:', min(uebrig))

Übrige Stücke je Startwert 0 bis 7 (Budget 3000):
cliquen3  kalt 0.05 -> 0.05  [2, 2, 6, 2, 2, 2, 2, 2]  kleinster Wert: 2
cliquen3  heiß 1 -> 0.05     [2, 2, 2, 2, 2, 2, 2, 2]  kleinster Wert: 2
cliquen5  kalt 0.05 -> 0.05  [19, 19, 33, 19, 19, 19, 19, 19]  kleinster Wert: 19
cliquen5  heiß 1 -> 0.05     [19, 19, 19, 19, 19, 19, 19, 19]  kleinster Wert: 19


#### Frage E1.3

1. Mit der **heißen** Suche kommen alle 8 Läufe auf denselben Wert, mit der **kalten** nicht. Lies die Werte ab: Wie viele Läufe bleiben bei der kalten Suche **über** dem besten Wert, und was ist das? (`paeckchen3`: kleinster Wert 2, `paeckchen5`: 19.)
2. Die Hinweise des Wettbewerbs und das Lösungsnotebook nennen **2** übrige Stücke bei `paeckchen3` und **19** bei `paeckchen5`. Sind das auch unsere kleinsten Werte? Kann die lokale Suche beweisen, dass man nicht besser sein kann?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Bei der kalten Suche bleiben 1 von 8 Läufen bei `paeckchen3` und 1 von 8 bei `paeckchen5` über dem besten Wert: Sie stecken in einem **lokalen Optimum** (eine Verschlechterung wäre nötig, um herauszukommen). Die heiße Suche kommt bei allen 16 Läufen auf den kleinsten Wert.
2. Ja: 2 und 19. Beweisen kann die lokale Suche das **nicht**: Sie findet nur Lösungen, nie eine Schranke. Das übernimmt der Solver im nächsten Abschnitt.

</details>

## Zum Schluss: Der Beweis mit MiniZinc

Die lokale Suche findet 2 und 19, aber sie **beweist** nicht, dass das Optimum ist. Ein **Solver** kann das, weil das Modell über die Cliquen **linear** ist. Du kennst von den MiniZinc-Arbeitsblättern 2D-Arrays, `sum` mit `where`, `forall` und `.dzn`-Dateien. Neu ist höchstens das **Array mit drei Indizes** `x[k, s, r]`: Wie viele Stücke der Sorte `s` und des Stils `r` liegen in Boxen der Clique `k`? Dazu eine Zahl `boxen[k]` der Boxen je Clique. Bedingungen: Je Clique und Sorte liegen zwischen `boxen[k]` und `3 · boxen[k]` Stücke, von jedem Stück werden nicht mehr verwendet, als da sind, und Stile außerhalb der Clique sind verboten. Maximiert wird die Zahl der gepackten Stücke. **MiniZinc läuft nicht in JupyterLite:** Kopiere das Modell in die MiniZinc IDE oder in den [MiniZinc Playground](https://play.minizinc.org/) und füge die Daten als `.dzn` dazu (die Python-Zelle darunter erzeugt sie). Als Löser nimmst du **HiGHS** (ein Solver für ganzzahlige lineare Programme; in der MiniZinc IDE unter *Solver configuration* wählbar), der Playground bietet andere Löser an.

```minizinc
int: S;                                   % Sorten
int: R;                                   % Stile
int: K;                                   % maximale Cliquen
array[1..K, 1..R] of 0..1: in_clique;     % in_clique[k, r] = 1, wenn Stil r zur Clique k gehört
array[1..S, 1..R] of int: anzahl;         % Stücke je Sorte und Stil

array[1..K] of var 0..sum(anzahl): boxen;                 % Boxen je Clique
array[1..K, 1..S, 1..R] of var 0..max(anzahl): x;         % Stücke (s, r) in Boxen der Clique k

constraint forall(k in 1..K, s in 1..S, r in 1..R where in_clique[k, r] == 0)(x[k, s, r] = 0);
constraint forall(k in 1..K, s in 1..S)(
  sum(r in 1..R)(x[k, s, r]) >= boxen[k] /\ sum(r in 1..R)(x[k, s, r]) <= 3 * boxen[k]);
constraint forall(s in 1..S, r in 1..R)(sum(k in 1..K)(x[k, s, r]) <= anzahl[s, r]);

solve maximize sum(x);
output ["uebrig = ", show(sum(anzahl) - sum(x)), "\n"];
```

Die Python-Zelle erzeugt die Datendatei:

In [ ]:
def als_dzn(dateiname):
    sorten, stile, cliquen, anzahl = lies_cliquen(dateiname)
    in_clique = []
    for clique in cliquen:
        for r in range(stile):
            in_clique.append(1 if r in clique else 0)
    zahlen = []
    for zeile in anzahl:
        zahlen += zeile
    return (f'S = {sorten};\nR = {stile};\nK = {len(cliquen)};\n'
            f'in_clique = array2d(1..{len(cliquen)}, 1..{stile}, {in_clique});\n'
            f'anzahl = array2d(1..{sorten}, 1..{stile}, {zahlen});')


print(als_dzn('daten/paeckchen_cliquen3.txt'))

S = 5;
R = 10;
K = 6;
in_clique = array2d(1..6, 1..10, [1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1]);
anzahl = array2d(1..5, 1..10, [4, 2, 0, 3, 0, 0, 3, 4, 2, 0, 4, 4, 0, 0, 11, 1, 1, 2, 1, 1, 2, 1, 2, 6, 0, 2, 0, 0, 2, 1, 2, 6, 1, 0, 6, 1, 0, 0, 4, 0, 0, 1, 1, 5, 0, 3, 0, 4, 2, 1]);


Mit HiGHS ist das Modell bei beiden Dateien in kürzester Zeit fertig (gemessen beim Erzeugen dieses Notebooks, Zeitlimit 60 Sekunden; `==========` heißt: Optimum bewiesen). Für `paeckchen_cliquen5.txt`:

**Ausgabe:**

```
uebrig = 19
----------
==========
```

Für `paeckchen_cliquen3.txt` meldet der Solver **2** übrige Stücke. Damit steht fest:

- Die **2** bei `paeckchen3` und die **19** bei `paeckchen5` sind **optimal**, genau die Werte der lokalen Suche und des Lösungsnotebooks. Das Lösungsnotebook verwendet dieses Modell auch für alle sieben nichtleeren Beispiele der Aufgabe.
- Die **kalte** Suche, die bei 6 bzw. 33 übrigen Stücken steckenblieb, war also wirklich **nicht optimal**. Ohne den Maßstab hätten wir das nicht gewusst.

#### Frage E1.4

1. Was ist der Vorteil der lokalen Suche, was der Vorteil des Solvers hier?
2. Was ist die **Güte** der kalten Suche in der Tabelle (schlechtester Lauf bei `paeckchen5`, 33 übrig statt 19)? Rechne in Prozent der **gepackten** Stücke (174 Stücke insgesamt).

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Die lokale Suche braucht **keine Struktur** der Aufgabe, jede Bewertung und Nachbarschaft genügen, sie skaliert auch auf Aufgaben, bei denen der Solver nicht fertig wird. Der Solver **beweist die Optimalität**, hier sofort, weil das Modell durch die Clique-Idee klein und **linear** ist. Beides zusammen ist ideal: Die Suche liefert gute Lösungen auf **allen** Größen, der Solver den **Maßstab** auf den kleinen.
2. Optimal sind 174 − 19 = 155 gepackte Stücke, der schlechteste Lauf packt 174 − 33 = 141: **91,0 %** des Optimums. Über alle 8 Läufe der kalten Suche liegt das Mittel höher; wichtig ist, dass ein einzelner Lauf **eine ganze Klasse schlechter** sein kann.

</details>

## Wettbewerbs-Ecke 2 (für Schnelle): Kool

*Angelehnt an Aufgabe 2 „Kool“ der 2. Runde des 30. Bundeswettbewerbs Informatik (2011/12).*

Aus **Würfeln mit Gängen** (Geradeaus-Gänge, Knicke, T-Kreuzungen, Kreuzungen, Sackgassen) soll ein **Labyrinth** auf einem rechteckigen **Brett** gebaut werden. Die Würfel dürfen **nicht gedreht** werden (ihre Farbmarkierung zeigt bei allen in dieselbe Richtung). Die Gänge müssen **zusammenpassen**: Hat ein Würfel an einer Seite eine Öffnung, muss der Nachbar an der gegenüberliegenden Seite auch eine haben, und **nach außen** darf nur der **Eingang** offen sein. Außerdem muss jeder Würfel **vom Eingang aus erreichbar** sein. Das Lösungsnotebook [Kool](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/30_Kool/kool_loesung.ipynb) behandelt die Aufgabe ganz, mit Tiefensuche und Simulated Annealing.

Wir vereinfachen stark: Wir nehmen die **26 Würfel des Aufgabenblatts** (`daten/kool_wuerfel.txt`, jedes Kastenzeichen ist ein Würfel: `─` links und rechts offen, `┌` rechts und unten offen, `╴` nur links offen und so weiter) und das Brett **5 × 6** (vier Felder bleiben frei). Der Eingang liegt **links vom Feld oben links**. Ein Zustand ist die Belegung des Bretts, ein **Zug** vertauscht zwei Felder (ein freies Feld zählt wie ein Würfel ohne Gänge), und die Bewertung ist **minus die Zahl der falschen Kanten**: Eine Kante zwischen zwei Feldern (oder zwischen Feld und Außenwelt) ist **falsch**, wenn nur eine Seite offen ist. Die Erreichbarkeit prüfen wir **nur am Ende**, weil sie bei jedem Schritt viel Rechenzeit kostet. Nach einem Tausch ändern sich nur die Kanten rund um die zwei Felder. Das ist wieder ein Fall für die inkrementelle Rechnung.

Der Rest (Würfel einlesen, Kanten zählen, Erreichbarkeit, Zufallsstart, die Funktionen für `abkuehlen`) steht in dieser Zelle fertig da. Die Idee der Funktion `fehler_um`: Zu einem Feld gehören **vier Kanten** (rechts, unten, und die rechte Kante des linken sowie die untere Kante des oberen Nachbarn), und jede Kante prüft, ob **beide Seiten** offen oder beide geschlossen sind. Beim Tausch zweier Felder müssen also nur **höchstens acht** Kanten neu angesehen werden.

In [ ]:
SCHRITT = {'L': (0, -1), 'R': (0, 1), 'O': (-1, 0), 'U': (1, 0)}      # so ändern sich (Zeile, Spalte)
GEGEN = {'L': 'R', 'R': 'L', 'O': 'U', 'U': 'O'}
ZEICHEN = {'LR': '─', 'OU': '│', 'LU': '┐', 'RO': '└', 'RU': '┌', 'LO': '┘', 'LOU': '┤', 'LRO': '┴', 'ROU': '├',
           'LRU': '┬', 'LROU': '┼', 'L': '╴', 'O': '╵', 'R': '╶', 'U': '╷'}
WUERFEL = {}                                    # Zeichen -> offene Seiten (L, R, O, U)
for seiten in ZEICHEN:
    WUERFEL[ZEICHEN[seiten]] = seiten


def lies_wuerfel(dateiname):
    wuerfel = []
    with open(dateiname, encoding='utf-8') as datei:
        for zeile in datei:
            if zeile.startswith('#'):
                continue
            for zeichen in zeile:
                if zeichen in WUERFEL:
                    wuerfel.append(WUERFEL[zeichen])
    return wuerfel


def zeige(brett):
    text = []
    for zeile in brett:
        text.append(''.join(ZEICHEN[w] if w != '' else '·' for w in zeile))
    return '\n'.join(text)


def kontakt(brett, r, c, d):
    """Ist das Feld (r, c) an der Seite d offen? Außerhalb des Bretts ist nur links neben (0, 0) offen (der Eingang)."""
    if 0 <= r < len(brett) and 0 <= c < len(brett[0]):
        return d in brett[r][c]
    return (r, c, d) == (0, -1, 'R')


def fehler_um(brett, felder):
    """Zahl der falschen Kanten rund um die angegebenen Felder (jede Kante nur einmal)."""
    kanten = set()
    for r, c in felder:
        kanten.add((r, c, 'R'))                 # rechts
        kanten.add((r, c, 'U'))                 # unten
        kanten.add((r, c - 1, 'R'))             # links = rechts vom linken Nachbarn
        kanten.add((r - 1, c, 'U'))             # oben = unten vom oberen Nachbarn
    anzahl = 0
    for r, c, d in kanten:
        r2 = r + SCHRITT[d][0]
        c2 = c + SCHRITT[d][1]
        if kontakt(brett, r, c, d) != kontakt(brett, r2, c2, GEGEN[d]):
            anzahl += 1
    return anzahl


def zusammenhang(brett):
    """Zahl der Würfel, die man vom Eingangsfeld (0, 0) aus durch die Gänge erreicht (Tiefensuche)."""
    m, n = len(brett), len(brett[0])
    gesehen = {(0, 0)}
    stapel = [(0, 0)]
    while len(stapel) > 0:
        r, c = stapel.pop()
        for d in brett[r][c]:
            r2 = r + SCHRITT[d][0]
            c2 = c + SCHRITT[d][1]
            if 0 <= r2 < m and 0 <= c2 < n and GEGEN[d] in brett[r2][c2] and (r2, c2) not in gesehen:
                gesehen.add((r2, c2))
                stapel.append((r2, c2))
    return len(gesehen)


def neuer_zustand_kool(wuerfel, m, n, zufall):
    inhalt = wuerfel + [''] * (m * n - len(wuerfel))        # '' ist ein freies Feld
    for i in range(len(inhalt) - 1, 0, -1):                 # mischen
        j = zufall.randint(0, i)
        inhalt[i], inhalt[j] = inhalt[j], inhalt[i]
    brett = []
    for r in range(m):
        brett.append(inhalt[r * n:(r + 1) * n])
    return {'m': m, 'n': n, 'brett': brett}


def wert_kool(z):
    felder = []
    for r in range(z['m']):
        for c in range(z['n']):
            felder.append((r, c))
    return -fehler_um(z['brett'], felder)


def zufallszug_kool(z, zufall):
    while True:
        r1 = zufall.randint(0, z['m'] - 1)
        c1 = zufall.randint(0, z['n'] - 1)
        r2 = zufall.randint(0, z['m'] - 1)
        c2 = zufall.randint(0, z['n'] - 1)
        if z['brett'][r1][c1] != z['brett'][r2][c2]:        # gleiche Würfel zu tauschen bringt nichts
            return ((r1, c1), (r2, c2))


def anwenden_kool(z, zug):
    (r1, c1), (r2, c2) = zug
    brett = z['brett']
    brett[r1][c1], brett[r2][c2] = brett[r2][c2], brett[r1][c1]


def kopie_kool(z):
    return {'m': z['m'], 'n': z['n'], 'brett': [zeile[:] for zeile in z['brett']]}

#### Aufgabe E2.1

Schreibe `delta_kool(z, zug)`: Der Zug ist `((r1, c1), (r2, c2))`, die Bewertung ist **minus** die Zahl der falschen Kanten. Zähle die falschen Kanten rund um die beiden Felder **vorher** (`fehler_um(z['brett'], list(zug))`), führe den Zug aus (`anwenden_kool`), zähle **nachher**, **tausche zurück** (derselbe Aufruf noch einmal) und gib `vorher - nachher` zurück (ein Verlust an falschen Kanten ist ein Gewinn an Wert). Der Test prüft die Funktion an 200 Zügen gegen die ganz neu berechnete Bewertung.

**Erwartete Ausgabe:**
```
26 Würfel auf einem 5 x 6 Brett, falsche Kanten beim Zufallsstart: 36
delta stimmt bei 200 Zügen: True
```

In [ ]:
def delta_kool(z, zug):
    felder = list(zug)
    # dein Code
    return 0


PROBLEM_KOOL = {'wert': wert_kool, 'zufallszug': zufallszug_kool, 'delta': delta_kool,
                'anwenden': anwenden_kool, 'kopie': kopie_kool}

wuerfel = lies_wuerfel('daten/kool_wuerfel.txt')
z = neuer_zustand_kool(wuerfel, 5, 6, random.Random(0))
print(len(wuerfel), 'Würfel auf einem 5 x 6 Brett, falsche Kanten beim Zufallsstart:', -wert_kool(z))
zufall = random.Random(1)
ok = True
for _ in range(200):
    zug = zufallszug_kool(z, zufall)
    d = delta_kool(z, zug)
    vorher = wert_kool(z)
    anwenden_kool(z, zug)
    ok = ok and d == wert_kool(z) - vorher
print('delta stimmt bei 200 Zügen:', ok)

<details>
<summary>Tipp anzeigen</summary>

Ein Tausch ist seine eigene Umkehrung: Zweimal derselbe Zug stellt den alten Zustand wieder her.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def delta_kool(z, zug):
    felder = list(zug)
    vorher = fehler_um(z['brett'], felder)
    anwenden_kool(z, zug)
    nachher = fehler_um(z['brett'], felder)
    anwenden_kool(z, zug)                       # zurücktauschen
    return vorher - nachher


PROBLEM_KOOL = {'wert': wert_kool, 'zufallszug': zufallszug_kool, 'delta': delta_kool,
                'anwenden': anwenden_kool, 'kopie': kopie_kool}

wuerfel = lies_wuerfel('daten/kool_wuerfel.txt')
z = neuer_zustand_kool(wuerfel, 5, 6, random.Random(0))
print(len(wuerfel), 'Würfel auf einem 5 x 6 Brett, falsche Kanten beim Zufallsstart:', -wert_kool(z))
zufall = random.Random(1)
ok = True
for _ in range(200):
    zug = zufallszug_kool(z, zufall)
    d = delta_kool(z, zug)
    vorher = wert_kool(z)
    anwenden_kool(z, zug)
    ok = ok and d == wert_kool(z) - vorher
print('delta stimmt bei 200 Zügen:', ok)
```

</details>

#### Aufgabe E2.2

Starte die Suche mit den Startwerten 0, 1, 2, … (Budget 20 000, Starttemperatur 1, Endtemperatur 0,1) und höre beim **ersten** Labyrinth auf, das **keine** falschen Kanten hat **und** bei dem alle Würfel vom Eingang aus erreichbar sind. Die Zelle ist fertig, führe sie aus und lies die Ausgabe. Bei jedem Startwert wird eine **neue** Suche gemacht (ein Neustart).

**Erwartete Ausgabe:**
```
Startwert 0: falsche Kanten 0, vom Eingang erreichbar 19 von 26
Startwert 1: falsche Kanten 0, vom Eingang erreichbar 26 von 26
─┐┌─┐·
·├┘·├┐
╷└─┬┘│
│┌┬┼┬┘
└┘└┴┘·
```

In [ ]:
for startwert in range(6):
    zufall = random.Random(startwert)
    z = neuer_zustand_kool(wuerfel, 5, 6, zufall)
    z2, wert = abkuehlen(PROBLEM_KOOL, z, 20000, 1, faktor_fuer(1, 0.1, 20000), zufall)
    erreicht = zusammenhang(z2['brett'])
    print(f'Startwert {startwert}: falsche Kanten {-wert}, vom Eingang erreichbar {erreicht} von {len(wuerfel)}')
    if wert == 0 and erreicht == len(wuerfel):
        print(zeige(z2['brett']))
        break

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
for startwert in range(6):
    zufall = random.Random(startwert)
    z = neuer_zustand_kool(wuerfel, 5, 6, zufall)
    z2, wert = abkuehlen(PROBLEM_KOOL, z, 20000, 1, faktor_fuer(1, 0.1, 20000), zufall)
    erreicht = zusammenhang(z2['brett'])
    print(f'Startwert {startwert}: falsche Kanten {-wert}, vom Eingang erreichbar {erreicht} von {len(wuerfel)}')
    if wert == 0 and erreicht == len(wuerfel):
        print(zeige(z2['brett']))
        break
```

</details>

#### Frage E2.3

1. Beim ersten Startwert sind alle Kanten in Ordnung, aber nicht alle Würfel erreichbar. Was bedeutet das für die Aufgabe, und was bedeutet es für die Bewertung?
2. Warum wird die Erreichbarkeit nicht schon in der Bewertung mitgezählt? Was würde sich ändern, wenn man sie mitzählt?
3. Das Programm hört beim ersten Treffer auf. Was haben die Startwerte mit **Neustarts** zu tun, und was beweist ein Treffer (oder kein Treffer) über die Aufgabe?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Die Bewertung „keine falschen Kanten“ ist **notwendig, aber nicht hinreichend**: Das Labyrinth kann aus **mehreren Teilen** bestehen, von denen nur einer vom Eingang erreichbar ist. Die Bewertung ist also **unvollständig**, wie ein Schiedsrichter, der eine Regel vergisst. Deshalb wird am Ende **zusätzlich** geprüft. Das ist hier der Fall bei Startwert 0 (siehe Ausgabe).
2. Die Erreichbarkeit zu berechnen ist **viel teurer** als die acht Kanten (Tiefensuche über alle 30 Felder statt nur die Kanten rund um zwei Felder, also O(Felder) statt O(1)). Das Lösungsnotebook zeigt einen Trick: Es prüft die Erreichbarkeit nur, wenn die Kanten allein schon gut genug sind. Wir lassen es hier weg und **schauen am Ende** nach, dazu muss man bei einem Misserfolg neu starten.
3. Jeder Startwert ist ein **Neustart** der Suche mit einem anderen Zufallsstart. Ein **Treffer** ist ein **Beweis durch Beispiel**: Es **gibt** ein Labyrinth auf dem Brett 5 × 6 (der Schiedsrichter `zusammenhang` und die Kantenzählung bestätigen es). **Kein Treffer** beweist nichts: Vielleicht gibt es keines, vielleicht fehlt nur Glück oder Budget. Ob es auf einem Brett **kein** Labyrinth gibt, kann man mit der lokalen Suche nie zeigen, dafür braucht man vollständige Suche (Backtracking, Notebook 06) oder einen Solver.

</details>

**MiniZinc:** Das Lösungsnotebook rechnet die Aufgabe auch mit MiniZinc. Es passt hier **nur eingeschränkt**: Die Kantenbedingungen sind leicht aufzuschreiben, aber die **Erreichbarkeit vom Eingang** braucht ein umständliches Hilfsmodell, und die Solver sind **unberechenbar** (derselbe Solver, der 210 Würfel in unter einer Sekunde unterbringt, findet für 92 Würfel in einer Minute nichts; Messtabelle im [Lösungsnotebook](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/30_Kool/kool_loesung.ipynb)). Für diese Ecke lassen wir es weg.

## Das hast du gelernt

```python
# Eine allgemeine lokale Suche: die Aufgabe steckt in fünf Funktionen
def abkuehlen(problem, z, budget, t0, faktor, zufall):
    z = problem['kopie'](z)
    wert = problem['wert'](z)
    bester_wert = wert
    bester_z = problem['kopie'](z)
    t = t0
    for schritt in range(budget):
        zug = problem['zufallszug'](z, zufall)
        d = problem['delta'](z, zug)                          # inkrementell, nicht die ganze Bewertung
        if d >= 0 or zufall.random() < math.exp(d / t):
            problem['anwenden'](z, zug)
            wert += d
            if wert > bester_wert:
                bester_wert = wert
                bester_z = problem['kopie'](z)
        t = t * faktor
    return bester_z, bester_wert
```

- Eine lokale Suche besteht aus **Zustand, Bewertung, Zug** und einem **Start**. Du schreibst die Aufgabenfunktionen (`wert`, `zufallszug`, `delta`, `anwenden`, `kopie`) und gibst sie der allgemeinen Suchfunktion als **Funktionen im Wörterbuch**: Dieselbe `abkuehlen` bearbeitete Arbeitsgruppen, Päckchen und Kool.
- **Nachbarschaften:** Verschieben, Tauschen und größere Züge (Ringe). Größere Nachbarschaften haben weniger lokale Optima, aber teurere Schritte (O(n), O(n²), O(n³) Züge).
- **Inkrementell bewerten:** Nur die Teile neu rechnen, die der Zug berührt. Das spart den Faktor n (hier: 435 Paare gegen 29 bis 84), man muss die Formel aber mit einem **Schiedsrichter** gegen die ganze Bewertung prüfen (Finde den Fehler: Tauschen).
- **Bergsteigen** (erste oder beste Verbesserung) endet in einem **lokalen Optimum**, das **von der Nachbarschaft abhängt**. Ein Lauf reicht nicht: Die Streuung ist groß.
- **Neustarts** und **Simulated Annealing** (Starttemperatur, Abkühlfaktor, Budget) kommen aus lokalen Optima heraus. Die Temperatur hat die **Einheit der Bewertung**.
- **Parameter durch Experiment:** dasselbe **Budget** für alle (in Schritten, nicht Sekunden), mehrere Startwerte, **Mittel und Streuung**, ein Unterschied zählt nur, wenn er größer als die Streuung ist, und nachprüfen an **anderen** Eingaben.
- **Gierig starten** lohnt sich vor allem bei knappem Budget und kalter Suche. Die **Güte** misst man an der Schranke, am **Optimum kleiner Fälle** (Durchprobieren oder Solver) und an der besten Lösung langer Läufe, und man sagt **ehrlich**, was bewiesen ist und was nicht.

### Aufgaben der 2. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Stilvolle Päckchen](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/42_StilvollePaeckchen/paeckchen_loesung.ipynb) (42. BwInf): Zuordnung zu Cliquen, lokale Suche und MiniZinc als Beweis
- [Kool](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/30_Kool/kool_loesung.ipynb) (30.): Labyrinth aus Würfeln, Tiefensuche und Simulated Annealing
- [Ned](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/30_Ned/ned_loesung.ipynb) (30.): Bergsteigen mit Störungen („iterierte lokale Suche“), Vergleich mit dem exakten Plan
- [Konfetti](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/43_Konfetti/konfetti_loesung.ipynb) (43.): Bergsteigen mit Neustarts, Beweis mit Branch and Bound

Weitere Aufgaben mit lokaler Suche stehen in der Werkzeugübersicht der 2. Runde (Abschnitt „Heuristiken“). Das nächste Kern-Notebook ist 06 „Backtracking für große Fälle“: Wenn du die **beste** Lösung sicher brauchst und die lokale Suche sie nur „wahrscheinlich“ findet. Zwischendurch gibt es ein Themen-Notebook 05 „Touren“, das die lokale Suche auf Rundreisen anwendet (2-opt).